# load dependecies

In [ ]:
## Initialzing and loading required libraries and subfunctions
import numpy as np
import matplotlib.pyplot as plt
import copy
import yasa
from mne.filter import resample
import pynapple as nap
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import normalize
import requests
from io import BytesIO
import sails
from ssqueezepy import cwt
import re
from scipy.stats import entropy
import os, sys

import scipy
from scipy import signal
from scipy.interpolate import griddata
from scipy.signal import correlate
from scipy.stats import pearsonr
from scipy.fft import fft
from scipy.spatial.distance import euclidean
from scipy.signal import spectrogram
from scipy.io import loadmat
import scipy.fft
import scipy.stats
import scipy.io as sio
from scipy.signal import hilbert

import emd as emd
import emd.sift as sift
import emd.spectra as spectra

from neurodsp.sim import sim_combined
from neurodsp.plts import plot_time_series, plot_timefrequency
from neurodsp.utils import create_times
from neurodsp.timefrequency.wavelets import compute_wavelet_transform
from neurodsp.filt import filter_signal

# Load required libraries
import numpy as np
from scipy.io import loadmat
from scipy.signal import hilbert
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import seaborn as sns
from neurodsp.filt import filter_signal, filter_signal_fir, design_fir_filter
import emd
import pandas as pd
from sklearn.preprocessing import Normalizer
from tqdm import tqdm
import plotly.express as px
import copy
import umap.umap_ as umap
from scipy.spatial import cKDTree
import pickle
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable

## UTILS

for rel in ('src', '../src'):
    p = os.path.abspath(os.path.join(os.getcwd(), rel))
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

from utils import *
from detect_pt import *

from scipy.io import loadmat
import numpy as np
from neurodsp.filt import filter_signal
import copy
import emd
from scipy.spatial import cKDTree
from tqdm import tqdm

sns.set(style='white', context='notebook')

In [ ]:
path_to_config = '/Users/amir/Desktop/for Abdel/emd_masksift_CA1_config.yml'
config = emd.sift.SiftConfig.from_yaml_file(path_to_config)

# Preprocessing

In [ ]:
# preprocessing
def extract_cycle_info(imfs, imf_frequencies):

  waveforms = pd.DataFrame()
  all_trials = pd.DataFrame()
  raw_wavelets = []
  all_FPPs = []

  theta_range = [5, 12]
  frequencies = np.arange(15, 141, 1)  # Logarithmically spaced frequencies from 15 to 300 Hz
  angles=np.linspace(-180,180,19)
  fs = 1000

  for idx, imf in enumerate(imfs):
    cycle_data = get_cycle_data(imf[:, 5], fs)

    amp_thresh = np.percentile(cycle_data['IA'], 25) # higher than 25th percentile of the data
    lo_freq_duration = fs/5  # restrict the analysis to 5-12 Hz
    hi_freq_duration = fs/12

    conditions = ['is_good==1',
                        f'duration_samples<{lo_freq_duration}',
                        f'duration_samples>{hi_freq_duration}',
                        f'max_amp>{amp_thresh}']
    print(len(cycle_data['theta_imf']))
    all_cycles = get_cycles_with_conditions(cycle_data['cycles'], conditions)
    
    # Check if any cycles satisfy the conditions
    if all_cycles is None or all_cycles.chain_vect.size == 0:
        print("No cycles satisfy the conditions.")
        return pd.DataFrame(), pd.DataFrame(), []
    
    subset_cycles_df = all_cycles.get_metric_dataframe(subset=True)
    subset_indices = subset_cycles_df['index'].values

    all_cycles_inds = get_cycle_inds(all_cycles, subset_indices)
    cycles_inds = arrange_cycle_inds(all_cycles_inds)

    freqs = imf_frequencies[idx]
    sub_theta, theta, supra_theta = tg_split(freqs, theta_range)
    supra_theta_sig = np.sum(imf.T[supra_theta], axis=0)

    # # Corrected Wavelet Transform Computation
    raw_data=sails.wavelet.morlet(supra_theta_sig, freqs=frequencies, sample_rate=fs, ncycles=5,ret_mode='power', normalise='simple')
    raw_wavelets.append(raw_data)
    supraPlot = scipy.stats.zscore(raw_data, axis=1)
    FPP = bin_tf_to_fpp(cycles_inds, supraPlot, bin_count=19)
    all_FPPs.append(FPP)

    # Compute mode frequency for each cycle
    mode_freqs, entropies = compute_mode_frequency_and_entropy(FPP, frequencies, angles)

    all_waveforms, _ = emd.cycles.phase_align(cycle_data['IP'], cycle_data['theta_imf'],
                                                            cycles=all_cycles.iterate(through='subset'), npoints=100)
    all_waveforms = pd.DataFrame(all_waveforms.T)

    waveforms = pd.concat([waveforms, all_waveforms])

    trial = all_cycles.get_metric_dataframe(subset=True)
    trial['mode_freqs'] = mode_freqs
    trial['entropy'] = entropies
    all_trials = pd.concat([all_trials, trial])

  return waveforms, all_trials, all_FPPs

In [ ]:
path_to_hpc = '/Users/amir/Desktop/for Abdel/RGS/DatabyCondition/HomeCageHC/OS_Ephys_RGS14_Rat1_57986_SD4_HC_01_08_2018/2018-08-01_15-49-15_Post-Trial5/HPC_100_CH4_0.continuous.mat'
path_to_pfc = '/Users/amir/Desktop/for Abdel/RGS/DatabyCondition/HomeCageHC/OS_Ephys_RGS14_Rat1_57986_SD4_HC_01_08_2018/2018-08-01_15-49-15_Post-Trial5/PFC_100_CH47_0.continuous.mat'
path_to_hypno = '/Users/amir/Desktop/for Abdel/RGS/DatabyCondition/HomeCageHC/OS_Ephys_RGS14_Rat1_57986_SD4_HC_01_08_2018/2018-08-01_15-49-15_Post-Trial5/2018-08-01_15-49-15_Post-Trial5_concatenated-states.mat'

In [ ]:
fs = 1000
theta_range = [5, 12]

In [ ]:
lfpHPC, hypno, _ = get_data(path_to_hpc, path_to_hypno)

In [ ]:
lfpPFC, _, _ = get_data(path_to_pfc, path_to_hypno, type='pfc')

In [ ]:
phasic_interval, tonic_interval, lfp_raw = extract_pt_intervals(lfpHPC, hypno, fs=1000)

In [ ]:
tonic_imfs, tonic_freqs, _ = extract_imfs_by_pt_intervals(
                        lfp_raw, fs, tonic_interval, config, return_imfs_freqs=True)
phasic_imfs, phasic_freqs, _ = extract_imfs_by_pt_intervals(
                        lfp_raw, fs, phasic_interval, config, return_imfs_freqs=True)

In [ ]:
tonic_theta_imfs  = [imf[:, 5] for imf in tonic_imfs]
phasic_theta_imfs = [imf[:, 5] for imf in phasic_imfs]

In [ ]:
phasic_waveforms, phasic_trials, phasic_fpps = extract_cycle_info(phasic_imfs, phasic_freqs)
tonic_waveforms, tonic_trials, tonic_fpps = extract_cycle_info(tonic_imfs, tonic_freqs)

In [ ]:
def extract_lfp_by_pt_intervals(lfp, fs, interval):
    rem_lfp = []
    for ii in range(len(interval)):
        start_idx = int(interval.loc[ii, 'start'] * fs)
        end_idx = int(interval.loc[ii, 'end'] * fs)
        rem_lfp.append(np.array(lfp[start_idx:end_idx]))
    return rem_lfp


In [ ]:
pfc_phasic_rem_lfp = extract_lfp_by_pt_intervals(lfpPFC, fs, phasic_interval)
pfc_tonic_rem_lfp = extract_lfp_by_pt_intervals(lfpPFC, fs, tonic_interval)

In [ ]:
theta_pfc_phasic_rem_lfp = []
for sig in pfc_phasic_rem_lfp:
    filtered = bandpass_filter(sig, fs=1000, low=15, high=200, order=4)
    theta_pfc_phasic_rem_lfp.append(filtered)

theta_pfc_tonic_rem_lfp = []
for sig in pfc_tonic_rem_lfp:
    filtered = bandpass_filter(sig, fs=1000, low=15, high=200, order=4)
    theta_pfc_tonic_rem_lfp.append(filtered)

# HPC-PFC Theta-Cycle Gamma Connectivity Pipeline
**Goal**: For each valid HPC theta cycle, extract gamma activity from both HPC (supra-theta IMFs) and PFC (bandpass filtered), then compute cycle-by-cycle connectivity measures.

## Step 1: Extract theta cycle indices and supra-theta signals from HPC

In [ ]:
def extract_cycle_indices_and_gamma(imfs, imf_frequencies):
    """
    Extract theta cycle indices and supra-theta (gamma) signals from HPC IMFs.
    
    Returns:
        all_cycle_inds: list of Nx2 arrays [start, end] per interval
        all_supra_theta_sigs: list of supra-theta signals per interval
    """
    all_cycle_inds = []
    all_supra_theta_sigs = []
    
    theta_range = [5, 12]
    fs = 1000
    
    for idx, imf in enumerate(imfs):
        cycle_data = get_cycle_data(imf[:, 5], fs)
        
        amp_thresh = np.percentile(cycle_data['IA'], 25)
        lo_freq_duration = fs / 5
        hi_freq_duration = fs / 12
        
        conditions = ['is_good==1',
                      f'duration_samples<{lo_freq_duration}',
                      f'duration_samples>{hi_freq_duration}',
                      f'max_amp>{amp_thresh}']
        
        all_cycles = get_cycles_with_conditions(cycle_data['cycles'], conditions)
        
        if all_cycles is None or all_cycles.chain_vect.size == 0:
            all_cycle_inds.append(np.array([]).reshape(0, 2).astype(int))
            all_supra_theta_sigs.append(np.zeros(imf.shape[0]))
            continue
        
        subset_cycles_df = all_cycles.get_metric_dataframe(subset=True)
        subset_indices = subset_cycles_df['index'].values
        cycle_inds_list = get_cycle_inds(all_cycles, subset_indices)
        cycle_inds = arrange_cycle_inds(cycle_inds_list)
        all_cycle_inds.append(cycle_inds)
        
        # Extract supra-theta (gamma) signal from IMFs
        freqs = imf_frequencies[idx]
        sub_theta, theta, supra_theta = tg_split(freqs, theta_range)
        supra_theta_sig = np.sum(imf.T[supra_theta], axis=0)
        all_supra_theta_sigs.append(supra_theta_sig)
    
    return all_cycle_inds, all_supra_theta_sigs

In [ ]:
# Extract cycle indices and HPC gamma for phasic and tonic intervals
phasic_cycle_inds, phasic_hpc_gamma = extract_cycle_indices_and_gamma(phasic_imfs, phasic_freqs)
tonic_cycle_inds, tonic_hpc_gamma = extract_cycle_indices_and_gamma(tonic_imfs, tonic_freqs)

print(f"Phasic: {len(phasic_cycle_inds)} intervals, "
      f"{sum(len(c) for c in phasic_cycle_inds)} total cycles")
print(f"Tonic: {len(tonic_cycle_inds)} intervals, "
      f"{sum(len(c) for c in tonic_cycle_inds)} total cycles")

## Step 2: Compute complex wavelet transforms for HPC and PFC gamma

We use complex Morlet wavelets to get both **phase** and **amplitude** at each frequency (15-140 Hz) for both regions. HPC gamma comes from supra-theta IMFs; PFC gamma comes from the bandpass-filtered LFP.

In [ ]:
import pywt

frequencies = np.arange(15, 141, 1)  # same frequency range as in extract_cycle_info
fs = 1000
n_cycles_wavelet = 5

# Complex Morlet wavelet parameters for pywt
# cmor(B, C): B = bandwidth, C = center frequency
# To match n_cycles oscillations: B = n_cycles^2 / (2 * pi^2 * C^2)
C = 1.0
B = n_cycles_wavelet**2 / (2 * np.pi**2 * C**2)
wavelet_name = f'cmor{B:.4f}-{C:.4f}'

# Convert desired frequencies to scales: scale = C * fs / freq
scales = C * fs / frequencies

print(f"Wavelet: {wavelet_name}")
print(f"Scales range: {scales.min():.2f} - {scales.max():.2f}")
print(f"Frequencies: {frequencies[0]} - {frequencies[-1]} Hz")

def compute_complex_wavelets(signals, frequencies, fs, n_cycles=5):
    """
    Compute complex Morlet wavelet transforms using PyWavelets.
    Returns list of complex arrays (n_freqs, n_timepoints).
    """
    C = 1.0
    B = n_cycles**2 / (2 * np.pi**2 * C**2)
    wavelet = f'cmor{B:.4f}-{C:.4f}'
    sc = C * fs / frequencies
    
    all_wavelets = []
    for sig in signals:
        coeffs, _ = pywt.cwt(sig, sc, wavelet, sampling_period=1.0/fs)
        all_wavelets.append(coeffs)  # (n_freqs, n_timepoints) complex
    return all_wavelets

In [ ]:
# Complex wavelets for HPC gamma (supra-theta IMF signals)
print("Computing HPC phasic wavelets...")
phasic_hpc_cwt = compute_complex_wavelets(phasic_hpc_gamma, frequencies, fs, n_cycles=n_cycles_wavelet)
print("Computing HPC tonic wavelets...")
tonic_hpc_cwt = compute_complex_wavelets(tonic_hpc_gamma, frequencies, fs, n_cycles=n_cycles_wavelet)

# Complex wavelets for PFC gamma (bandpass-filtered signals)
print("Computing PFC phasic wavelets...")
phasic_pfc_cwt = compute_complex_wavelets(theta_pfc_phasic_rem_lfp, frequencies, fs, n_cycles=n_cycles_wavelet)
print("Computing PFC tonic wavelets...")
tonic_pfc_cwt = compute_complex_wavelets(theta_pfc_tonic_rem_lfp, frequencies, fs, n_cycles=n_cycles_wavelet)

print("Done.")

## Step 3: Connectivity Functions

Three connectivity measures computed **per theta cycle, per frequency**:

1. **Phase Locking Value (PLV)**: Consistency of phase difference between HPC and PFC gamma across time points within each cycle. High PLV = stable phase relationship.
2. **Amplitude Envelope Correlation**: Pearson correlation of HPC and PFC gamma amplitudes within each cycle. Measures co-fluctuation of gamma power.
3. **Coherence (magnitude-squared)**: Combines phase and amplitude consistency; computed from the cross-spectral density within each cycle.

In [ ]:
def compute_plv_per_cycle(hpc_cwt, pfc_cwt, cycle_inds):
    """
    Compute Phase Locking Value between HPC and PFC gamma for each theta cycle.
    
    PLV = |mean(exp(i * (phase_hpc - phase_pfc)))| over time within cycle.
    
    Args:
        hpc_cwt: complex wavelet (n_freqs, n_timepoints)
        pfc_cwt: complex wavelet (n_freqs, n_timepoints)
        cycle_inds: Nx2 array of [start, end] indices
    Returns:
        plv: (n_cycles, n_freqs) array
    """
    n_cycles = cycle_inds.shape[0]
    n_freqs = hpc_cwt.shape[0]
    plv = np.full((n_cycles, n_freqs), np.nan)
    
    hpc_phase = np.angle(hpc_cwt)
    pfc_phase = np.angle(pfc_cwt)
    
    for i in range(n_cycles):
        start, end = cycle_inds[i]
        if end - start < 2:
            continue
        phase_diff = hpc_phase[:, start:end] - pfc_phase[:, start:end]
        plv[i] = np.abs(np.mean(np.exp(1j * phase_diff), axis=1))
    
    return plv


def compute_amp_corr_per_cycle(hpc_cwt, pfc_cwt, cycle_inds):
    """
    Compute amplitude envelope correlation between HPC and PFC gamma per cycle.
    
    Args:
        hpc_cwt: complex wavelet (n_freqs, n_timepoints)
        pfc_cwt: complex wavelet (n_freqs, n_timepoints)
        cycle_inds: Nx2 array of [start, end] indices
    Returns:
        amp_corr: (n_cycles, n_freqs) array of Pearson r values
    """
    n_cycles = cycle_inds.shape[0]
    n_freqs = hpc_cwt.shape[0]
    amp_corr = np.full((n_cycles, n_freqs), np.nan)
    
    hpc_amp = np.abs(hpc_cwt)
    pfc_amp = np.abs(pfc_cwt)
    
    for i in range(n_cycles):
        start, end = cycle_inds[i]
        if end - start < 4:  # need enough points for correlation
            continue
        for f in range(n_freqs):
            h = hpc_amp[f, start:end]
            p = pfc_amp[f, start:end]
            # skip if either signal is constant
            if np.std(h) == 0 or np.std(p) == 0:
                continue
            amp_corr[i, f], _ = pearsonr(h, p)
    
    return amp_corr


def compute_coherence_per_cycle(hpc_cwt, pfc_cwt, cycle_inds):
    """
    Compute magnitude-squared coherence between HPC and PFC gamma per cycle.
    
    Coherence = |<S_xy>|^2 / (<S_xx> * <S_yy>) computed from wavelet coefficients
    within each theta cycle.
    
    Args:
        hpc_cwt: complex wavelet (n_freqs, n_timepoints)
        pfc_cwt: complex wavelet (n_freqs, n_timepoints)
        cycle_inds: Nx2 array of [start, end] indices
    Returns:
        coh: (n_cycles, n_freqs) array
    """
    n_cycles = cycle_inds.shape[0]
    n_freqs = hpc_cwt.shape[0]
    coh = np.full((n_cycles, n_freqs), np.nan)
    
    for i in range(n_cycles):
        start, end = cycle_inds[i]
        if end - start < 2:
            continue
        hpc_seg = hpc_cwt[:, start:end]
        pfc_seg = pfc_cwt[:, start:end]
        
        # Cross-spectral density
        Sxy = np.mean(hpc_seg * np.conj(pfc_seg), axis=1)
        Sxx = np.mean(np.abs(hpc_seg)**2, axis=1)
        Syy = np.mean(np.abs(pfc_seg)**2, axis=1)
        
        denom = Sxx * Syy
        valid = denom > 0
        coh[i, valid] = np.abs(Sxy[valid])**2 / denom[valid]
    
    return coh


def compute_all_connectivity(hpc_cwt_list, pfc_cwt_list, cycle_inds_list):
    """
    Compute all connectivity measures across all intervals.
    
    Returns:
        plv_all: concatenated PLV (total_cycles, n_freqs)
        amp_corr_all: concatenated amplitude correlation (total_cycles, n_freqs)
        coh_all: concatenated coherence (total_cycles, n_freqs)
    """
    plv_all, amp_corr_all, coh_all = [], [], []
    
    for hpc_cwt, pfc_cwt, cycle_inds in zip(hpc_cwt_list, pfc_cwt_list, cycle_inds_list):
        if len(cycle_inds) == 0:
            continue
        
        plv = compute_plv_per_cycle(hpc_cwt, pfc_cwt, cycle_inds)
        amp_corr = compute_amp_corr_per_cycle(hpc_cwt, pfc_cwt, cycle_inds)
        coh = compute_coherence_per_cycle(hpc_cwt, pfc_cwt, cycle_inds)
        
        plv_all.append(plv)
        amp_corr_all.append(amp_corr)
        coh_all.append(coh)
    
    return (np.vstack(plv_all) if plv_all else np.array([]),
            np.vstack(amp_corr_all) if amp_corr_all else np.array([]),
            np.vstack(coh_all) if coh_all else np.array([]))

## Step 4: Compute Connectivity for Phasic and Tonic REM

In [ ]:
print("Computing phasic connectivity...")
phasic_plv, phasic_amp_corr, phasic_coh = compute_all_connectivity(
    phasic_hpc_cwt, phasic_pfc_cwt, phasic_cycle_inds)

print("Computing tonic connectivity...")
tonic_plv, tonic_amp_corr, tonic_coh = compute_all_connectivity(
    tonic_hpc_cwt, tonic_pfc_cwt, tonic_cycle_inds)

print(f"\nPhasic: {phasic_plv.shape[0]} cycles x {phasic_plv.shape[1]} freqs")
print(f"Tonic:  {tonic_plv.shape[0]} cycles x {tonic_plv.shape[1]} freqs")

## Step 5: Compute Phase-Resolved Connectivity (FPP-style)

Bin connectivity measures by theta phase (same 19-bin approach as FPP) to see how HPC-PFC gamma connectivity varies across the theta cycle.

In [ ]:
def compute_phase_resolved_connectivity(hpc_cwt_list, pfc_cwt_list, cycle_inds_list, bin_count=19):
    """
    Compute connectivity binned by theta phase within each cycle (like FPP).
    
    For each cycle, the time samples are divided into `bin_count` phase bins.
    PLV and coherence are computed within each phase bin across cycles.
    
    Returns:
        plv_fpp: (n_freqs, bin_count) - mean PLV resolved by theta phase
        coh_fpp: (n_freqs, bin_count) - mean coherence resolved by theta phase
        amp_corr_fpp: (n_freqs, bin_count) - mean amplitude product resolved by theta phase
    """
    n_freqs = hpc_cwt_list[0].shape[0]
    
    # Accumulate phase differences and amplitudes per phase bin
    phase_diff_bins = [[[] for _ in range(bin_count)] for _ in range(n_freqs)]
    amp_product_bins = [[[] for _ in range(bin_count)] for _ in range(n_freqs)]
    
    for hpc_cwt, pfc_cwt, cycle_inds in zip(hpc_cwt_list, pfc_cwt_list, cycle_inds_list):
        if len(cycle_inds) == 0:
            continue
            
        hpc_phase = np.angle(hpc_cwt)
        pfc_phase = np.angle(pfc_cwt)
        hpc_amp = np.abs(hpc_cwt)
        pfc_amp = np.abs(pfc_cwt)
        
        for i in range(cycle_inds.shape[0]):
            start, end = cycle_inds[i]
            n_samples = end - start
            if n_samples < bin_count:
                continue
            
            # Assign each time point to a phase bin (evenly spaced across cycle)
            bin_assignments = np.linspace(0, bin_count, n_samples, endpoint=False).astype(int)
            bin_assignments = np.clip(bin_assignments, 0, bin_count - 1)
            
            for b in range(bin_count):
                mask = bin_assignments == b
                if not np.any(mask):
                    continue
                for f in range(n_freqs):
                    pd = hpc_phase[f, start:end][mask] - pfc_phase[f, start:end][mask]
                    phase_diff_bins[f][b].extend(pd)
                    ap = hpc_amp[f, start:end][mask] * pfc_amp[f, start:end][mask]
                    amp_product_bins[f][b].extend(ap)
    
    # Compute PLV and mean amplitude product per bin
    plv_fpp = np.full((n_freqs, bin_count), np.nan)
    amp_fpp = np.full((n_freqs, bin_count), np.nan)
    
    for f in range(n_freqs):
        for b in range(bin_count):
            if len(phase_diff_bins[f][b]) > 0:
                pd_arr = np.array(phase_diff_bins[f][b])
                plv_fpp[f, b] = np.abs(np.mean(np.exp(1j * pd_arr)))
                amp_fpp[f, b] = np.mean(amp_product_bins[f][b])
    
    return plv_fpp, amp_fpp

In [ ]:
print("Computing phase-resolved connectivity (phasic)...")
phasic_plv_fpp, phasic_amp_fpp = compute_phase_resolved_connectivity(
    phasic_hpc_cwt, phasic_pfc_cwt, phasic_cycle_inds)

print("Computing phase-resolved connectivity (tonic)...")
tonic_plv_fpp, tonic_amp_fpp = compute_phase_resolved_connectivity(
    tonic_hpc_cwt, tonic_pfc_cwt, tonic_cycle_inds)

print("Done.")

## Step 6: Visualization

In [ ]:
# --- Plot 1: Mean connectivity spectra (PLV, Amp Corr, Coherence) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# PLV spectrum
ax = axes[0]
phasic_plv_mean = np.nanmean(phasic_plv, axis=0)
tonic_plv_mean = np.nanmean(tonic_plv, axis=0)
phasic_plv_sem = np.nanstd(phasic_plv, axis=0) / np.sqrt(np.sum(~np.isnan(phasic_plv), axis=0))
tonic_plv_sem = np.nanstd(tonic_plv, axis=0) / np.sqrt(np.sum(~np.isnan(tonic_plv), axis=0))

ax.plot(frequencies, phasic_plv_mean, color='crimson', label='Phasic')
ax.fill_between(frequencies, phasic_plv_mean - phasic_plv_sem, phasic_plv_mean + phasic_plv_sem,
                color='crimson', alpha=0.2)
ax.plot(frequencies, tonic_plv_mean, color='steelblue', label='Tonic')
ax.fill_between(frequencies, tonic_plv_mean - tonic_plv_sem, tonic_plv_mean + tonic_plv_sem,
                color='steelblue', alpha=0.2)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PLV')
ax.set_title('Phase Locking Value: HPC-PFC Gamma')
ax.legend()

# Amplitude correlation spectrum
ax = axes[1]
phasic_ac_mean = np.nanmean(phasic_amp_corr, axis=0)
tonic_ac_mean = np.nanmean(tonic_amp_corr, axis=0)
phasic_ac_sem = np.nanstd(phasic_amp_corr, axis=0) / np.sqrt(np.sum(~np.isnan(phasic_amp_corr), axis=0))
tonic_ac_sem = np.nanstd(tonic_amp_corr, axis=0) / np.sqrt(np.sum(~np.isnan(tonic_amp_corr), axis=0))

ax.plot(frequencies, phasic_ac_mean, color='crimson', label='Phasic')
ax.fill_between(frequencies, phasic_ac_mean - phasic_ac_sem, phasic_ac_mean + phasic_ac_sem,
                color='crimson', alpha=0.2)
ax.plot(frequencies, tonic_ac_mean, color='steelblue', label='Tonic')
ax.fill_between(frequencies, tonic_ac_mean - tonic_ac_sem, tonic_ac_mean + tonic_ac_sem,
                color='steelblue', alpha=0.2)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Pearson r')
ax.set_title('Amplitude Correlation: HPC-PFC Gamma')
ax.legend()

# Coherence spectrum
ax = axes[2]
phasic_coh_mean = np.nanmean(phasic_coh, axis=0)
tonic_coh_mean = np.nanmean(tonic_coh, axis=0)
phasic_coh_sem = np.nanstd(phasic_coh, axis=0) / np.sqrt(np.sum(~np.isnan(phasic_coh), axis=0))
tonic_coh_sem = np.nanstd(tonic_coh, axis=0) / np.sqrt(np.sum(~np.isnan(tonic_coh), axis=0))

ax.plot(frequencies, phasic_coh_mean, color='crimson', label='Phasic')
ax.fill_between(frequencies, phasic_coh_mean - phasic_coh_sem, phasic_coh_mean + phasic_coh_sem,
                color='crimson', alpha=0.2)
ax.plot(frequencies, tonic_coh_mean, color='steelblue', label='Tonic')
ax.fill_between(frequencies, tonic_coh_mean - tonic_coh_sem, tonic_coh_mean + tonic_coh_sem,
                color='steelblue', alpha=0.2)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Coherence')
ax.set_title('Magnitude-Squared Coherence: HPC-PFC Gamma')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 2: Phase-resolved PLV (FPP-style) ---
phase_extent = (-180, 180)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, fpp, label in zip(axes, [phasic_plv_fpp, tonic_plv_fpp], ['Phasic', 'Tonic']):
    # Mode frequency: frequency with highest PLV across phase bins
    plv_per_freq = np.nanmax(fpp, axis=1)
    mode_idx = int(np.nanargmax(plv_per_freq))
    mode_freq = float(frequencies[mode_idx])

    im = ax.imshow(
        fpp, aspect='auto', origin='lower', interpolation='bilinear',
        extent=[phase_extent[0], phase_extent[1], frequencies[0], frequencies[-1]],
        cmap='hot',
    )
    ax.axhline(mode_freq, color='cyan', linestyle='--', linewidth=1.5,
               label=f'Mode {mode_freq:.1f} Hz')
    plt.colorbar(im, ax=ax, label='PLV')
    ax.set_xlabel('Theta Phase (deg)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(f'{label} REM | Phase-Resolved PLV (HPC-PFC)')
    ax.legend(loc='upper right', frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 3: Phase-resolved amplitude coupling (FPP-style) ---
phasic_amp_fpp_z = scipy.stats.zscore(phasic_amp_fpp, axis=1)
tonic_amp_fpp_z = scipy.stats.zscore(tonic_amp_fpp, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, fpp, label in zip(axes, [phasic_amp_fpp_z, tonic_amp_fpp_z], ['Phasic', 'Tonic']):
    amp_per_freq = np.nanmax(fpp, axis=1)
    mode_idx = int(np.nanargmax(amp_per_freq))
    mode_freq = float(frequencies[mode_idx])

    im = ax.imshow(
        fpp, aspect='auto', origin='lower', interpolation='bilinear',
        extent=[phase_extent[0], phase_extent[1], frequencies[0], frequencies[-1]],
        cmap='hot',
    )
    ax.axhline(mode_freq, color='cyan', linestyle='--', linewidth=1.5,
               label=f'Mode {mode_freq:.1f} Hz')
    plt.colorbar(im, ax=ax, label='Z-scored amp product')
    ax.set_xlabel('Theta Phase (deg)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(f'{label} REM | Phase-Resolved Amp Coupling (HPC-PFC)')
    ax.legend(loc='upper right', frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 4: Difference plots (Phasic - Tonic) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# PLV difference
ax = axes[0]
plv_diff = phasic_plv_fpp - tonic_plv_fpp
vmax = np.nanmax(np.abs(plv_diff))
diff_per_freq = np.nanmax(plv_diff, axis=1)
mode_idx = int(np.nanargmax(diff_per_freq))
mode_freq = float(frequencies[mode_idx])

im = ax.imshow(
    plv_diff, aspect='auto', origin='lower', interpolation='bilinear',
    extent=[phase_extent[0], phase_extent[1], frequencies[0], frequencies[-1]],
    cmap='RdBu_r', vmin=-vmax, vmax=vmax,
)
ax.axhline(mode_freq, color='cyan', linestyle='--', linewidth=1.5,
           label=f'Peak diff {mode_freq:.1f} Hz')
plt.colorbar(im, ax=ax, label='\u0394PLV')
ax.set_xlabel('Theta Phase (deg)')
ax.set_ylabel('Frequency (Hz)')
ax.set_title('PLV Difference (Phasic \u2212 Tonic)')
ax.legend(loc='upper right', frameon=False)

# Amplitude coupling difference
ax = axes[1]
amp_diff = phasic_amp_fpp_z - tonic_amp_fpp_z
vmax = np.nanmax(np.abs(amp_diff))
diff_per_freq = np.nanmax(amp_diff, axis=1)
mode_idx = int(np.nanargmax(diff_per_freq))
mode_freq = float(frequencies[mode_idx])

im = ax.imshow(
    amp_diff, aspect='auto', origin='lower', interpolation='bilinear',
    extent=[phase_extent[0], phase_extent[1], frequencies[0], frequencies[-1]],
    cmap='RdBu_r', vmin=-vmax, vmax=vmax,
)
ax.axhline(mode_freq, color='cyan', linestyle='--', linewidth=1.5,
           label=f'Peak diff {mode_freq:.1f} Hz')
plt.colorbar(im, ax=ax, label='\u0394Z-scored amp')
ax.set_xlabel('Theta Phase (deg)')
ax.set_ylabel('Frequency (Hz)')
ax.set_title('Amp Coupling Difference (Phasic \u2212 Tonic)')
ax.legend(loc='upper right', frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 5: Distribution of per-cycle broadband PLV (averaged across frequencies) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Broadband PLV distribution
phasic_plv_broadband = np.nanmean(phasic_plv, axis=1)
tonic_plv_broadband = np.nanmean(tonic_plv, axis=1)

ax = axes[0]
ax.hist(phasic_plv_broadband[~np.isnan(phasic_plv_broadband)], bins=30, alpha=0.6,
        color='crimson', label='Phasic', density=True)
ax.hist(tonic_plv_broadband[~np.isnan(tonic_plv_broadband)], bins=30, alpha=0.6,
        color='steelblue', label='Tonic', density=True)
ax.set_xlabel('Mean PLV (15-140 Hz)')
ax.set_ylabel('Density')
ax.set_title('Per-Cycle Broadband PLV Distribution')
ax.legend()

# Broadband coherence distribution
phasic_coh_broadband = np.nanmean(phasic_coh, axis=1)
tonic_coh_broadband = np.nanmean(tonic_coh, axis=1)

ax = axes[1]
ax.hist(phasic_coh_broadband[~np.isnan(phasic_coh_broadband)], bins=30, alpha=0.6,
        color='crimson', label='Phasic', density=True)
ax.hist(tonic_coh_broadband[~np.isnan(tonic_coh_broadband)], bins=30, alpha=0.6,
        color='steelblue', label='Tonic', density=True)
ax.set_xlabel('Mean Coherence (15-140 Hz)')
ax.set_ylabel('Density')
ax.set_title('Per-Cycle Broadband Coherence Distribution')
ax.legend()

plt.tight_layout()
plt.show()

# Statistical test
from scipy.stats import mannwhitneyu
stat_plv, p_plv = mannwhitneyu(phasic_plv_broadband[~np.isnan(phasic_plv_broadband)],
                                tonic_plv_broadband[~np.isnan(tonic_plv_broadband)])
stat_coh, p_coh = mannwhitneyu(phasic_coh_broadband[~np.isnan(phasic_coh_broadband)],
                                tonic_coh_broadband[~np.isnan(tonic_coh_broadband)])
print(f"PLV: Phasic median={np.nanmedian(phasic_plv_broadband):.4f}, "
      f"Tonic median={np.nanmedian(tonic_plv_broadband):.4f}, "
      f"Mann-Whitney U p={p_plv:.4e}")
print(f"Coherence: Phasic median={np.nanmedian(phasic_coh_broadband):.4f}, "
      f"Tonic median={np.nanmedian(tonic_coh_broadband):.4f}, "
      f"Mann-Whitney U p={p_coh:.4e}")

In [ ]:
def compute_single_cycle_fpp(hpc_cwt, pfc_cwt, cycle_inds, cycle_idx, bin_count=19):
    """
    Compute PLV and amplitude coupling FPP for a single theta cycle.
    
    Returns:
        plv_fpp: (n_freqs, bin_count) - PLV per freq per phase bin
        amp_fpp: (n_freqs, bin_count) - amplitude product per freq per phase bin
    """
    start, end = cycle_inds[cycle_idx]
    n_samples = end - start
    n_freqs = hpc_cwt.shape[0]
    
    if n_samples < bin_count:
        return np.full((n_freqs, bin_count), np.nan), np.full((n_freqs, bin_count), np.nan)
    
    hpc_seg = hpc_cwt[:, start:end]
    pfc_seg = pfc_cwt[:, start:end]
    
    hpc_phase = np.angle(hpc_seg)
    pfc_phase = np.angle(pfc_seg)
    hpc_amp = np.abs(hpc_seg)
    pfc_amp = np.abs(pfc_seg)
    
    bin_assignments = np.linspace(0, bin_count, n_samples, endpoint=False).astype(int)
    bin_assignments = np.clip(bin_assignments, 0, bin_count - 1)
    
    plv_fpp = np.full((n_freqs, bin_count), np.nan)
    amp_fpp = np.full((n_freqs, bin_count), np.nan)
    
    for b in range(bin_count):
        mask = bin_assignments == b
        if not np.any(mask):
            continue
        phase_diff = hpc_phase[:, mask] - pfc_phase[:, mask]
        plv_fpp[:, b] = np.abs(np.mean(np.exp(1j * phase_diff), axis=1))
        amp_fpp[:, b] = np.mean(hpc_amp[:, mask] * pfc_amp[:, mask], axis=1)
    
    return plv_fpp, amp_fpp


def plot_cycle_fpps(hpc_cwt_list, pfc_cwt_list, cycle_inds_list, frequencies,
                    cycle_type='phasic', start=0, end=10, metric='plv'):
    """
    Plot FPP for individual theta cycles.
    
    Args:
        metric: 'plv' or 'amp'
    """
    phase_extent = (-180, 180)
    
    # Flatten across intervals: collect (interval_idx, cycle_idx) pairs
    all_cycles = []
    for iv_idx, cycle_inds in enumerate(cycle_inds_list):
        for c_idx in range(len(cycle_inds)):
            n_samples = cycle_inds[c_idx, 1] - cycle_inds[c_idx, 0]
            all_cycles.append((iv_idx, c_idx, n_samples))
    
    if not all_cycles:
        print("No cycles found.")
        return
    
    plot_range = range(start, min(end, len(all_cycles)))
    
    for i in plot_range:
        iv_idx, c_idx, n_samples = all_cycles[i]
        
        plv_fpp, amp_fpp = compute_single_cycle_fpp(
            hpc_cwt_list[iv_idx], pfc_cwt_list[iv_idx],
            cycle_inds_list[iv_idx], c_idx)
        
        if metric == 'plv':
            fpp = plv_fpp
            cbar_label = 'PLV'
        else:
            fpp = scipy.stats.zscore(amp_fpp, axis=1)
            cbar_label = 'Z-scored amp product'
        
        # Mode frequency
        val_per_freq = np.nanmax(fpp, axis=1)
        if np.all(~np.isfinite(val_per_freq)):
            print(f"Cycle {i}: no finite values")
            continue
        mode_idx = int(np.nanargmax(val_per_freq))
        mode_freq = float(frequencies[mode_idx])
        
        plt.figure(figsize=(5, 4))
        plt.imshow(
            fpp, aspect='auto', origin='lower', interpolation='bilinear',
            extent=[phase_extent[0], phase_extent[1], frequencies[0], frequencies[-1]],
            cmap='hot',
        )
        plt.axhline(mode_freq, color='cyan', linestyle='--', linewidth=1.5,
                     label=f'Mode {mode_freq:.1f} Hz')
        plt.colorbar(label=cbar_label)
        plt.xlabel('Theta Phase (deg)')
        plt.ylabel('Frequency (Hz)')
        plt.title(f'Cycle {i} | {cycle_type} | interval={iv_idx} | '
                  f'n_samples={n_samples}')
        plt.legend(loc='upper right', frameon=False)
        plt.tight_layout()
        plt.show()

In [ ]:
# Plot first 10 phasic cycles - PLV
plot_cycle_fpps(phasic_hpc_cwt, phasic_pfc_cwt, phasic_cycle_inds, frequencies,
                cycle_type='phasic', start=0, end=10, metric='plv')

In [ ]:
# Plot first 10 phasic cycles - Amplitude coupling
plot_cycle_fpps(phasic_hpc_cwt, phasic_pfc_cwt, phasic_cycle_inds, frequencies,
                cycle_type='phasic', start=0, end=10, metric='amp')

In [ ]:
# Plot first 10 tonic cycles - PLV
plot_cycle_fpps(tonic_hpc_cwt, tonic_pfc_cwt, tonic_cycle_inds, frequencies,
                cycle_type='tonic', start=0, end=10, metric='plv')

# Pairwise Phase Consistency (PPC)\n\n**PPC** is an unbiased estimator of the squared PLV. While PLV is positively biased for short time segments (like individual theta cycles ~100-200ms), PPC corrects for this.\n\nFormula: `PPC = (N * PLV² - 1) / (N - 1)` where N is the number of phase difference observations.\n\n- PPC = 0 → no consistent phase relationship (random)\n- PPC = 1 → perfect phase locking\n- PPC can be slightly negative (noise)\n\nWe compute:\n1. **Per-cycle PPC** (n_cycles × n_freqs) — cycle-by-cycle connectivity\n2. **Phase-resolved PPC** (FPP-style, n_freqs × n_bins) — how PPC varies across theta phase\n3. **Per-interval PPC** — separately for each phasic and tonic REM interval

In [ ]:
def compute_ppc_from_plv(plv, n_observations):
    """
    Compute PPC from pre-computed PLV values.
    PPC = (N * PLV^2 - 1) / (N - 1)
    
    Args:
        plv: PLV values (any shape)
        n_observations: number of phase difference samples used to compute each PLV (same shape or scalar)
    Returns:
        ppc: PPC values (same shape as plv)
    """
    ppc = (n_observations * plv**2 - 1) / (n_observations - 1)
    return ppc


def compute_ppc_per_cycle(hpc_cwt, pfc_cwt, cycle_inds):
    """
    Compute PPC between HPC and PFC gamma for each theta cycle.
    
    For each cycle, we compute PPC directly from phase differences:
    PPC = (2 / (N*(N-1))) * sum_{i<j} cos(theta_i - theta_j)
    Efficient formula: PPC = (N * PLV^2 - 1) / (N - 1)
    
    Args:
        hpc_cwt: complex wavelet (n_freqs, n_timepoints)
        pfc_cwt: complex wavelet (n_freqs, n_timepoints)
        cycle_inds: Nx2 array of [start, end] indices
    Returns:
        ppc: (n_cycles, n_freqs) array
        n_obs: (n_cycles,) array of observation counts per cycle
    """
    n_cycles = cycle_inds.shape[0]
    n_freqs = hpc_cwt.shape[0]
    ppc = np.full((n_cycles, n_freqs), np.nan)
    n_obs = np.zeros(n_cycles, dtype=int)
    
    hpc_phase = np.angle(hpc_cwt)
    pfc_phase = np.angle(pfc_cwt)
    
    for i in range(n_cycles):
        start, end = cycle_inds[i]
        N = end - start
        if N < 3:  # need at least 3 observations for meaningful PPC
            continue
        n_obs[i] = N
        phase_diff = hpc_phase[:, start:end] - pfc_phase[:, start:end]
        # PLV for this cycle
        plv = np.abs(np.mean(np.exp(1j * phase_diff), axis=1))
        # Convert to PPC (unbiased)
        ppc[i] = (N * plv**2 - 1) / (N - 1)
    
    return ppc, n_obs


def compute_phase_resolved_ppc(hpc_cwt_list, pfc_cwt_list, cycle_inds_list, bin_count=19):
    """
    Compute PPC binned by theta phase within each cycle (FPP-style).
    
    For each phase bin, pools all phase differences across all cycles,
    then computes PPC from the pooled data.
    
    Returns:
        ppc_fpp: (n_freqs, bin_count) - PPC resolved by theta phase
    """
    n_freqs = hpc_cwt_list[0].shape[0]
    
    # Accumulate phase differences per phase bin
    phase_diff_bins = [[[] for _ in range(bin_count)] for _ in range(n_freqs)]
    
    for hpc_cwt, pfc_cwt, cycle_inds in zip(hpc_cwt_list, pfc_cwt_list, cycle_inds_list):
        if len(cycle_inds) == 0:
            continue
        
        hpc_phase = np.angle(hpc_cwt)
        pfc_phase = np.angle(pfc_cwt)
        
        for i in range(cycle_inds.shape[0]):
            start, end = cycle_inds[i]
            n_samples = end - start
            if n_samples < bin_count:
                continue
            
            bin_assignments = np.linspace(0, bin_count, n_samples, endpoint=False).astype(int)
            bin_assignments = np.clip(bin_assignments, 0, bin_count - 1)
            
            for b in range(bin_count):
                mask = bin_assignments == b
                if not np.any(mask):
                    continue
                for f in range(n_freqs):
                    pd = hpc_phase[f, start:end][mask] - pfc_phase[f, start:end][mask]
                    phase_diff_bins[f][b].extend(pd)
    
    # Compute PPC from pooled phase differences
    ppc_fpp = np.full((n_freqs, bin_count), np.nan)
    for f in range(n_freqs):
        for b in range(bin_count):
            diffs = np.array(phase_diff_bins[f][b])
            N = len(diffs)
            if N < 3:
                continue
            plv = np.abs(np.mean(np.exp(1j * diffs)))
            ppc_fpp[f, b] = (N * plv**2 - 1) / (N - 1)
    
    return ppc_fpp


def compute_ppc_per_interval(hpc_cwt_list, pfc_cwt_list, cycle_inds_list):
    """
    Compute PPC separately for each REM interval (each phasic/tonic segment).
    
    Returns:
        interval_ppc: list of (n_cycles_in_interval, n_freqs) arrays
        interval_ppc_mean: list of (n_freqs,) arrays — mean PPC per interval
        interval_n_cycles: list of int — number of valid cycles per interval
    """
    interval_ppc = []
    interval_ppc_mean = []
    interval_n_cycles = []
    
    for hpc_cwt, pfc_cwt, cycle_inds in zip(hpc_cwt_list, pfc_cwt_list, cycle_inds_list):
        if len(cycle_inds) == 0:
            interval_ppc.append(np.array([]))
            interval_ppc_mean.append(np.full(hpc_cwt.shape[0], np.nan))
            interval_n_cycles.append(0)
            continue
        
        ppc, n_obs = compute_ppc_per_cycle(hpc_cwt, pfc_cwt, cycle_inds)
        interval_ppc.append(ppc)
        interval_ppc_mean.append(np.nanmean(ppc, axis=0))
        interval_n_cycles.append(np.sum(~np.isnan(ppc[:, 0])))
    
    return interval_ppc, interval_ppc_mean, interval_n_cycles


print("PPC functions defined.")

### Compute PPC: per-cycle and phase-resolved

In [ ]:
# --- Per-cycle PPC for phasic and tonic ---
print("Computing per-cycle PPC (phasic)...")
phasic_ppc_all = []
phasic_nobs_all = []
for hpc_cwt, pfc_cwt, cinds in zip(phasic_hpc_cwt, phasic_pfc_cwt, phasic_cycle_inds):
    ppc, nobs = compute_ppc_per_cycle(hpc_cwt, pfc_cwt, cinds)
    phasic_ppc_all.append(ppc)
    phasic_nobs_all.append(nobs)

# Concatenate across all intervals
phasic_ppc = np.vstack(phasic_ppc_all)
phasic_nobs = np.concatenate(phasic_nobs_all)

print("Computing per-cycle PPC (tonic)...")
tonic_ppc_all = []
tonic_nobs_all = []
for hpc_cwt, pfc_cwt, cinds in zip(tonic_hpc_cwt, tonic_pfc_cwt, tonic_cycle_inds):
    ppc, nobs = compute_ppc_per_cycle(hpc_cwt, pfc_cwt, cinds)
    tonic_ppc_all.append(ppc)
    tonic_nobs_all.append(nobs)

tonic_ppc = np.vstack(tonic_ppc_all)
tonic_nobs = np.concatenate(tonic_nobs_all)

print(f"Phasic PPC: {phasic_ppc.shape[0]} cycles x {phasic_ppc.shape[1]} freqs")
print(f"Tonic PPC:  {tonic_ppc.shape[0]} cycles x {tonic_ppc.shape[1]} freqs")

# --- Phase-resolved PPC (FPP-style) ---
print("\nComputing phase-resolved PPC (phasic)...")
phasic_ppc_fpp = compute_phase_resolved_ppc(phasic_hpc_cwt, phasic_pfc_cwt, phasic_cycle_inds)

print("Computing phase-resolved PPC (tonic)...")
tonic_ppc_fpp = compute_phase_resolved_ppc(tonic_hpc_cwt, tonic_pfc_cwt, tonic_cycle_inds)

# --- Per-interval PPC ---
print("\nComputing per-interval PPC (phasic)...")
phasic_interval_ppc, phasic_interval_ppc_mean, phasic_interval_n_cycles = \
    compute_ppc_per_interval(phasic_hpc_cwt, phasic_pfc_cwt, phasic_cycle_inds)

print("Computing per-interval PPC (tonic)...")
tonic_interval_ppc, tonic_interval_ppc_mean, tonic_interval_n_cycles = \
    compute_ppc_per_interval(tonic_hpc_cwt, tonic_pfc_cwt, tonic_cycle_inds)

print(f"\nPhasic: {len(phasic_interval_ppc)} intervals")
print(f"Tonic:  {len(tonic_interval_ppc)} intervals")
print("Done.")

### PPC Visualization

In [ ]:
# --- Plot 1: Mean PPC spectrum (phasic vs tonic) ---
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

phasic_ppc_mean = np.nanmean(phasic_ppc, axis=0)
tonic_ppc_mean = np.nanmean(tonic_ppc, axis=0)
phasic_ppc_sem = np.nanstd(phasic_ppc, axis=0) / np.sqrt(np.sum(~np.isnan(phasic_ppc), axis=0))
tonic_ppc_sem = np.nanstd(tonic_ppc, axis=0) / np.sqrt(np.sum(~np.isnan(tonic_ppc), axis=0))

ax.plot(frequencies, phasic_ppc_mean, color='crimson', label='Phasic')
ax.fill_between(frequencies, phasic_ppc_mean - phasic_ppc_sem, phasic_ppc_mean + phasic_ppc_sem,
                color='crimson', alpha=0.2)
ax.plot(frequencies, tonic_ppc_mean, color='steelblue', label='Tonic')
ax.fill_between(frequencies, tonic_ppc_mean - tonic_ppc_sem, tonic_ppc_mean + tonic_ppc_sem,
                color='steelblue', alpha=0.2)
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PPC')
ax.set_title('Pairwise Phase Consistency: HPC-PFC Gamma')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 2: Phase-resolved PPC (FPP-style) ---
phase_extent = (-180, 180)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, fpp, label in zip(axes, [phasic_ppc_fpp, tonic_ppc_fpp], ['Phasic', 'Tonic']):
    # Mode frequency: frequency with highest PPC across phase bins
    ppc_per_freq = np.nanmax(fpp, axis=1)
    mode_idx = int(np.nanargmax(ppc_per_freq))
    mode_freq = float(frequencies[mode_idx])
    
    im = ax.imshow(
        fpp, aspect='auto', origin='lower', interpolation='bilinear',
        extent=[phase_extent[0], phase_extent[1], frequencies[0], frequencies[-1]],
        cmap='hot',
    )
    ax.axhline(mode_freq, color='cyan', linestyle='--', linewidth=1.5,
               label=f'Mode {mode_freq:.1f} Hz')
    plt.colorbar(im, ax=ax, label='PPC')
    ax.set_xlabel('Theta Phase (deg)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(f'{label} REM | Phase-Resolved PPC (HPC-PFC)')
    ax.legend(loc='upper right', frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 3: PPC difference (Phasic - Tonic) ---
fig, ax = plt.subplots(1, 1, figsize=(7, 5))
ppc_diff = phasic_ppc_fpp - tonic_ppc_fpp
vmax = np.nanmax(np.abs(ppc_diff))

im = ax.imshow(
    ppc_diff, aspect='auto', origin='lower', interpolation='bilinear',
    extent=[-180, 180, frequencies[0], frequencies[-1]],
    cmap='hot', vmin=-vmax, vmax=vmax,
)
plt.colorbar(im, ax=ax, label='PPC difference')
ax.set_xlabel('Theta Phase (deg)')
ax.set_ylabel('Frequency (Hz)')
ax.set_title('Phase-Resolved PPC Difference (Phasic - Tonic)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 4: Per-cycle PPC distribution (histogram) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Broadband PPC (averaged across frequencies) per cycle
phasic_ppc_broadband = np.nanmean(phasic_ppc, axis=1)
tonic_ppc_broadband = np.nanmean(tonic_ppc, axis=1)

ax = axes[0]
ax.hist(phasic_ppc_broadband[~np.isnan(phasic_ppc_broadband)], bins=30, color='crimson',
        alpha=0.6, label=f'Phasic (n={np.sum(~np.isnan(phasic_ppc_broadband))})', density=True)
ax.hist(tonic_ppc_broadband[~np.isnan(tonic_ppc_broadband)], bins=30, color='steelblue',
        alpha=0.6, label=f'Tonic (n={np.sum(~np.isnan(tonic_ppc_broadband))})', density=True)
ax.axvline(0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Broadband PPC')
ax.set_ylabel('Density')
ax.set_title('Per-Cycle PPC Distribution (15-140 Hz mean)')
ax.legend()

# Per-interval mean PPC spectrum
ax = axes[1]
for i, (mean_ppc, nc) in enumerate(zip(phasic_interval_ppc_mean, phasic_interval_n_cycles)):
    if nc > 0:
        ax.plot(frequencies, mean_ppc, color='crimson', alpha=0.4, linewidth=0.8)
for i, (mean_ppc, nc) in enumerate(zip(tonic_interval_ppc_mean, tonic_interval_n_cycles)):
    if nc > 0:
        ax.plot(frequencies, mean_ppc, color='steelblue', alpha=0.4, linewidth=0.8)

# Add grand means
ax.plot(frequencies, np.nanmean(phasic_ppc, axis=0), color='crimson', linewidth=2, label='Phasic (mean)')
ax.plot(frequencies, np.nanmean(tonic_ppc, axis=0), color='steelblue', linewidth=2, label='Tonic (mean)')
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PPC')
ax.set_title('Per-Interval PPC Spectra (thin) + Grand Mean (thick)')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 5: PPC comparison PLV vs PPC side-by-side ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PLV spectrum
ax = axes[0]
ax.plot(frequencies, np.nanmean(phasic_plv, axis=0), color='crimson', label='Phasic')
ax.plot(frequencies, np.nanmean(tonic_plv, axis=0), color='steelblue', label='Tonic')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PLV')
ax.set_title('PLV (biased)')
ax.legend()

# PPC spectrum
ax = axes[1]
ax.plot(frequencies, np.nanmean(phasic_ppc, axis=0), color='crimson', label='Phasic')
ax.plot(frequencies, np.nanmean(tonic_ppc, axis=0), color='steelblue', label='Tonic')
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PPC')
ax.set_title('PPC (unbiased)')
ax.legend()

plt.suptitle('PLV vs PPC: Effect of bias correction', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Full Dataset PPC Analysis

Expand PPC computation across all rats, conditions, and trials.
The pipeline: Iterate over all rats/conditions/trials (same traversal as `extract_data_for_rat_fppsig`).

For each trial: load HPC + PFC, extract phasic/tonic intervals, compute IMFs, wavelets, and PPC.
Store per-interval PPC so you can inspect any single phasic/tonic interval for any rat.
Aggregate across rats for group-level comparisons

In [ ]:
def compute_ppc_for_trial(lfp_file, pfc_file, state_file, config, frequencies, fs=1000, n_cycles_wavelet=5):
    """
    Full PPC pipeline for a single trial: load data, extract intervals,
    compute IMFs, wavelets, and PPC.
    
    Returns:
        result: dict with per-interval PPC data, or None if trial is unusable
    """
    try:
        lfpHPC, hypno, _ = get_data(lfp_file, state_file)
        lfpPFC, _, _ = get_data(pfc_file, state_file, type='pfc')
    except Exception as e:
        print(f"  Failed to load data: {e}")
        return None
    
    try:
        phasic_interval, tonic_interval, lfp_raw = extract_pt_intervals(lfpHPC, hypno, fs=fs)
    except ValueError:
        print(f"  No REM sleep found.")
        return None
    
    if not phasic_interval or not tonic_interval:
        print(f"  No phasic or tonic intervals.")
        return None
    
    # Check if intervals are empty DataFrames
    if hasattr(phasic_interval, '__len__') and len(phasic_interval) == 0:
        print(f"  Empty phasic intervals.")
        return None
    if hasattr(tonic_interval, '__len__') and len(tonic_interval) == 0:
        print(f"  Empty tonic intervals.")
        return None
    
    # Extract IMFs
    try:
        phasic_imfs, phasic_freqs, _ = extract_imfs_by_pt_intervals(
            lfp_raw, fs, phasic_interval, config, return_imfs_freqs=True)
        tonic_imfs, tonic_freqs, _ = extract_imfs_by_pt_intervals(
            lfp_raw, fs, tonic_interval, config, return_imfs_freqs=True)
    except Exception as e:
        print(f"  IMF extraction failed: {e}")
        return None
    
    # Extract cycle indices and HPC gamma
    phasic_cycle_inds, phasic_hpc_gamma = extract_cycle_indices_and_gamma(phasic_imfs, phasic_freqs)
    tonic_cycle_inds, tonic_hpc_gamma = extract_cycle_indices_and_gamma(tonic_imfs, tonic_freqs)
    
    n_phasic_cycles = sum(len(c) for c in phasic_cycle_inds)
    n_tonic_cycles = sum(len(c) for c in tonic_cycle_inds)
    if n_phasic_cycles == 0 and n_tonic_cycles == 0:
        print(f"  No valid theta cycles.")
        return None
    
    # Extract and filter PFC LFP
    pfc_phasic = extract_lfp_by_pt_intervals(lfpPFC, fs, phasic_interval)
    pfc_tonic = extract_lfp_by_pt_intervals(lfpPFC, fs, tonic_interval)
    pfc_phasic_filt = [bandpass_filter(sig, fs=fs, low=15, high=200, order=4) for sig in pfc_phasic]
    pfc_tonic_filt = [bandpass_filter(sig, fs=fs, low=15, high=200, order=4) for sig in pfc_tonic]
    
    # Compute complex wavelets
    phasic_hpc_cwt = compute_complex_wavelets(phasic_hpc_gamma, frequencies, fs, n_cycles=n_cycles_wavelet)
    tonic_hpc_cwt = compute_complex_wavelets(tonic_hpc_gamma, frequencies, fs, n_cycles=n_cycles_wavelet)
    phasic_pfc_cwt = compute_complex_wavelets(pfc_phasic_filt, frequencies, fs, n_cycles=n_cycles_wavelet)
    tonic_pfc_cwt = compute_complex_wavelets(pfc_tonic_filt, frequencies, fs, n_cycles=n_cycles_wavelet)
    
    # Compute per-cycle PPC
    phasic_ppc_list = []
    phasic_nobs_list = []
    for hcwt, pcwt, cinds in zip(phasic_hpc_cwt, phasic_pfc_cwt, phasic_cycle_inds):
        ppc, nobs = compute_ppc_per_cycle(hcwt, pcwt, cinds)
        phasic_ppc_list.append(ppc)
        phasic_nobs_list.append(nobs)
    
    tonic_ppc_list = []
    tonic_nobs_list = []
    for hcwt, pcwt, cinds in zip(tonic_hpc_cwt, tonic_pfc_cwt, tonic_cycle_inds):
        ppc, nobs = compute_ppc_per_cycle(hcwt, pcwt, cinds)
        tonic_ppc_list.append(ppc)
        tonic_nobs_list.append(nobs)
    
    # Compute phase-resolved PPC
    phasic_ppc_fpp = compute_phase_resolved_ppc(phasic_hpc_cwt, phasic_pfc_cwt, phasic_cycle_inds)
    tonic_ppc_fpp = compute_phase_resolved_ppc(tonic_hpc_cwt, tonic_pfc_cwt, tonic_cycle_inds)
    
    return {
        # Per-interval data (list of arrays, one per interval)
        'phasic_ppc_per_interval': phasic_ppc_list,   # list of (n_cycles, n_freqs)
        'tonic_ppc_per_interval': tonic_ppc_list,
        'phasic_nobs_per_interval': phasic_nobs_list,
        'tonic_nobs_per_interval': tonic_nobs_list,
        # Concatenated across intervals
        'phasic_ppc': np.vstack(phasic_ppc_list) if phasic_ppc_list else np.array([]),
        'tonic_ppc': np.vstack(tonic_ppc_list) if tonic_ppc_list else np.array([]),
        # Phase-resolved (FPP)
        'phasic_ppc_fpp': phasic_ppc_fpp,
        'tonic_ppc_fpp': tonic_ppc_fpp,
        # Cycle counts
        'n_phasic_cycles': n_phasic_cycles,
        'n_tonic_cycles': n_tonic_cycles,
        'n_phasic_intervals': len(phasic_ppc_list),
        'n_tonic_intervals': len(tonic_ppc_list),
    }

print("compute_ppc_for_trial defined.")

In [ ]:
def run_ppc_dataset(rat_ids=None, frequencies=np.arange(15, 141, 1), fs=1000, n_cycles_wavelet=5):
    """
    Run PPC analysis across the full dataset.
    
    Args:
        rat_ids: list of rat IDs (ints) to process, or None for all rats.
                 e.g. [1, 3, 4] or None
        frequencies: frequency vector for wavelet transform
        fs: sampling rate
        n_cycles_wavelet: wavelet cycles parameter
    
    Returns:
        results: dict keyed by (rat_id, condition, sd, trial_number) -> trial result dict
        metadata: list of dicts with recording info for each trial
    """
    base_path = '/Users/amir/Desktop/for Abdel/RGS/DatabyCondition'
    cfg = config  # from cell 2
    
    preferred_conditions = ["HomeCageHC", "RandomCon", "OverlappingOR", "StableCondOD"]
    conditions = [c for c in preferred_conditions if os.path.isdir(os.path.join(base_path, c))]
    
    folder_re = re.compile(r'^OS_Ephys_RGS14_Rat(\d+)_\d+_SD(\d+)_([\w-]+)_([\d_-]+)$')
    
    results = {}
    metadata = []
    
    for condition_folder in conditions:
        condition_path = os.path.join(base_path, condition_folder)
        
        for rat_folder in sorted(os.listdir(condition_path)):
            full_path = os.path.join(condition_path, rat_folder)
            if not os.path.isdir(full_path):
                continue
            m = folder_re.match(rat_folder)
            if not m:
                continue
            
            rat_id = int(m.group(1))
            if rat_ids is not None and rat_id not in rat_ids:
                continue
            
            sd_number = m.group(2)
            condition = m.group(3)
            
            # Find post-trial folders
            post_trial_folders = sorted([
                f for f in os.listdir(full_path)
                if os.path.isdir(os.path.join(full_path, f)) and re.search(r'Post[-_]Trial\d+', f)
            ])
            
            for pt_folder in post_trial_folders:
                trial_path = os.path.join(full_path, pt_folder)
                trial_match = re.search(r'Post[-_]Trial(\d+)', pt_folder)
                if not trial_match:
                    continue
                trial_number = int(trial_match.group(1))
                
                # Find HPC, PFC, and state files
                hpc_file, pfc_file, state_file = None, None, None
                for fname in os.listdir(trial_path):
                    low = fname.lower()
                    if 'hpc_100' in low and low.endswith('.mat'):
                        hpc_file = os.path.join(trial_path, fname)
                    elif 'pfc_100' in low and low.endswith('.mat'):
                        pfc_file = os.path.join(trial_path, fname)
                    elif 'states' in low and low.endswith('.mat'):
                        state_file = os.path.join(trial_path, fname)
                
                if not hpc_file or not pfc_file or not state_file:
                    continue
                
                key = (rat_id, condition, int(sd_number), trial_number)
                print(f"\n--- Rat{rat_id} | {condition} | SD{sd_number} | Trial{trial_number} ---")
                
                result = compute_ppc_for_trial(
                    hpc_file, pfc_file, state_file, cfg, frequencies, fs, n_cycles_wavelet)
                
                if result is not None:
                    results[key] = result
                    meta = {
                        'rat_id': rat_id,
                        'condition': condition,
                        'sd': int(sd_number),
                        'trial': trial_number,
                        'key': key,
                        'n_phasic_cycles': result['n_phasic_cycles'],
                        'n_tonic_cycles': result['n_tonic_cycles'],
                        'n_phasic_intervals': result['n_phasic_intervals'],
                        'n_tonic_intervals': result['n_tonic_intervals'],
                    }
                    metadata.append(meta)
                    print(f"  OK: {result['n_phasic_cycles']} phasic, {result['n_tonic_cycles']} tonic cycles")
                else:
                    print(f"  SKIPPED")
    
    print(f"\n{'='*60}")
    print(f"Total recordings processed: {len(results)}")
    total_phasic = sum(r['n_phasic_cycles'] for r in results.values())
    total_tonic = sum(r['n_tonic_cycles'] for r in results.values())
    print(f"Total phasic cycles: {total_phasic}")
    print(f"Total tonic cycles: {total_tonic}")
    
    return results, metadata

print("run_ppc_dataset defined.")

### Run PPC on the dataset
Set `rat_ids` to choose which rats to process:
- `rat_ids=None` → all rats
- `rat_ids=[1, 3]` → only Rat1 and Rat3
- `rat_ids=[1]` → only Rat1

In [ ]:
# ---- CHOOSE RATS HERE ----
# Set to None for all rats, or a list like [1, 3, 4] for specific rats
selected_rats = [1, 3, 6, 9]  # e.g. [1, 3] or None for all

results, metadata = run_ppc_dataset(rat_ids=selected_rats, frequencies=frequencies)

### Aggregate and visualize full-dataset PPC

In [ ]:
# --- Aggregate all PPC across dataset ---
# Pool all per-cycle PPC values
all_phasic_ppc = []
all_tonic_ppc = []
# Per-rat mean PPC spectra (for rat-level stats)
rat_phasic_ppc_means = {}  # rat_id -> list of mean PPC spectra
rat_tonic_ppc_means = {}

for key, res in results.items():
    rat_id = key[0]
    if res['phasic_ppc'].size > 0 and res['phasic_ppc'].ndim == 2:
        all_phasic_ppc.append(res['phasic_ppc'])
        rat_phasic_ppc_means.setdefault(rat_id, []).append(np.nanmean(res['phasic_ppc'], axis=0))
    if res['tonic_ppc'].size > 0 and res['tonic_ppc'].ndim == 2:
        all_tonic_ppc.append(res['tonic_ppc'])
        rat_tonic_ppc_means.setdefault(rat_id, []).append(np.nanmean(res['tonic_ppc'], axis=0))

all_phasic_ppc = np.vstack(all_phasic_ppc) if all_phasic_ppc else np.array([])
all_tonic_ppc = np.vstack(all_tonic_ppc) if all_tonic_ppc else np.array([])

print(f"Total pooled phasic cycles: {all_phasic_ppc.shape[0] if all_phasic_ppc.size else 0}")
print(f"Total pooled tonic cycles:  {all_tonic_ppc.shape[0] if all_tonic_ppc.size else 0}")
print(f"Rats with data: {sorted(set(rat_phasic_ppc_means.keys()) | set(rat_tonic_ppc_means.keys()))}")

In [ ]:
# --- Plot 1: Grand-average PPC spectrum (all rats pooled) ---
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

grand_phasic_mean = np.nanmean(all_phasic_ppc, axis=0)
grand_tonic_mean = np.nanmean(all_tonic_ppc, axis=0)
grand_phasic_sem = np.nanstd(all_phasic_ppc, axis=0) / np.sqrt(np.sum(~np.isnan(all_phasic_ppc), axis=0))
grand_tonic_sem = np.nanstd(all_tonic_ppc, axis=0) / np.sqrt(np.sum(~np.isnan(all_tonic_ppc), axis=0))

ax.plot(frequencies, grand_phasic_mean, color='crimson', label='Phasic')
ax.fill_between(frequencies, grand_phasic_mean - grand_phasic_sem,
                grand_phasic_mean + grand_phasic_sem, color='crimson', alpha=0.2)
ax.plot(frequencies, grand_tonic_mean, color='steelblue', label='Tonic')
ax.fill_between(frequencies, grand_tonic_mean - grand_tonic_sem,
                grand_tonic_mean + grand_tonic_sem, color='steelblue', alpha=0.2)
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PPC')
ax.set_title(f'Grand-Average PPC: HPC-PFC Gamma (all rats, '
             f'n_phasic={all_phasic_ppc.shape[0]}, n_tonic={all_tonic_ppc.shape[0]})')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 2: Per-rat PPC spectra ---
all_rat_ids = sorted(set(rat_phasic_ppc_means.keys()) | set(rat_tonic_ppc_means.keys()))
n_rats = len(all_rat_ids)
fig, axes = plt.subplots(1, n_rats, figsize=(5 * n_rats, 4), squeeze=False)

for i, rid in enumerate(all_rat_ids):
    ax = axes[0, i]
    
    # Average across all trials for this rat
    if rid in rat_phasic_ppc_means and rat_phasic_ppc_means[rid]:
        phasic_stack = np.array(rat_phasic_ppc_means[rid])
        phasic_m = np.nanmean(phasic_stack, axis=0)
        phasic_s = np.nanstd(phasic_stack, axis=0) / np.sqrt(len(phasic_stack)) if len(phasic_stack) > 1 else np.zeros_like(phasic_m)
        ax.plot(frequencies, phasic_m, color='crimson', label=f'Phasic (n={len(phasic_stack)})')
        ax.fill_between(frequencies, phasic_m - phasic_s, phasic_m + phasic_s, color='crimson', alpha=0.2)
    
    if rid in rat_tonic_ppc_means and rat_tonic_ppc_means[rid]:
        tonic_stack = np.array(rat_tonic_ppc_means[rid])
        tonic_m = np.nanmean(tonic_stack, axis=0)
        tonic_s = np.nanstd(tonic_stack, axis=0) / np.sqrt(len(tonic_stack)) if len(tonic_stack) > 1 else np.zeros_like(tonic_m)
        ax.plot(frequencies, tonic_m, color='steelblue', label=f'Tonic (n={len(tonic_stack)})')
        ax.fill_between(frequencies, tonic_m - tonic_s, tonic_m + tonic_s, color='steelblue', alpha=0.2)
    
    ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PPC')
    ax.set_title(f'Rat {rid}')
    ax.legend(fontsize=8)

plt.suptitle('Per-Rat PPC Spectra (mean across trials)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 3: Grand-average phase-resolved PPC (FPP-style, pooled across all rats) ---
# Pool phase diffs from all trials for grand FPP
all_phasic_ppc_fpps = []
all_tonic_ppc_fpps = []
for key, res in results.items():
    if res['phasic_ppc_fpp'] is not None:
        all_phasic_ppc_fpps.append(res['phasic_ppc_fpp'])
    if res['tonic_ppc_fpp'] is not None:
        all_tonic_ppc_fpps.append(res['tonic_ppc_fpp'])

# Average FPPs across trials (weighted equally per trial)
grand_phasic_fpp = np.nanmean(np.array(all_phasic_ppc_fpps), axis=0) if all_phasic_ppc_fpps else None
grand_tonic_fpp = np.nanmean(np.array(all_tonic_ppc_fpps), axis=0) if all_tonic_ppc_fpps else None

phase_extent = (-180, 180)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Compute shared colorbar limits for phasic & tonic
vmin_shared, vmax_shared = None, None
for fpp in [grand_phasic_fpp, grand_tonic_fpp]:
    if fpp is not None:
        fmin, fmax = np.nanmin(fpp), np.nanmax(fpp)
        vmin_shared = fmin if vmin_shared is None else min(vmin_shared, fmin)
        vmax_shared = fmax if vmax_shared is None else max(vmax_shared, fmax)

for ax, fpp, label in zip(axes[:2], [grand_phasic_fpp, grand_tonic_fpp], ['Phasic', 'Tonic']):
    if fpp is None:
        ax.set_title(f'{label}: no data')
        continue
    ppc_per_freq = np.nanmax(fpp, axis=1)
    mode_idx = int(np.nanargmax(ppc_per_freq))
    mode_freq = float(frequencies[mode_idx])
    
    im = ax.imshow(fpp, aspect='auto', origin='lower', interpolation='bilinear',
                   extent=[phase_extent[0], phase_extent[1], frequencies[0], frequencies[-1]],
                   cmap='hot', vmin=vmin_shared, vmax=vmax_shared)
    ax.axhline(mode_freq, color='cyan', linestyle='--', linewidth=1.5,
               label=f'Mode {mode_freq:.1f} Hz')
    plt.colorbar(im, ax=ax, label='PPC')
    ax.set_xlabel('Theta Phase (deg)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(f'{label} REM | Grand-Average Phase-Resolved PPC')
    ax.legend(loc='upper right', frameon=False)

# Difference
if grand_phasic_fpp is not None and grand_tonic_fpp is not None:
    ax = axes[2]
    diff = grand_phasic_fpp - grand_tonic_fpp
    vmax_diff = np.nanmax(np.abs(diff))
    im = ax.imshow(diff, aspect='auto', origin='lower', interpolation='bilinear',
                   extent=[-180, 180, frequencies[0], frequencies[-1]],
                   cmap='RdBu_r', vmin=-vmax_diff, vmax=vmax_diff)
    plt.colorbar(im, ax=ax, label='PPC difference')
    ax.set_xlabel('Theta Phase (deg)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title('Phasic - Tonic (Grand Average)')

plt.tight_layout()
plt.show()

### Inspect individual intervals\n\nUse `inspect_interval()` to view PPC for a specific phasic or tonic interval from any rat/trial.

In [ ]:
def list_available_intervals(results, metadata, rat_id=None):
    """Print all available intervals for browsing."""
    for meta in metadata:
        if rat_id is not None and meta['rat_id'] != rat_id:
            continue
        key = meta['key']
        res = results[key]
        print(f"Rat{meta['rat_id']} | {meta['condition']} | SD{meta['sd']} | Trial{meta['trial']} "
              f"| phasic: {meta['n_phasic_intervals']} intervals ({meta['n_phasic_cycles']} cycles) "
              f"| tonic: {meta['n_tonic_intervals']} intervals ({meta['n_tonic_cycles']} cycles)")


def inspect_interval(results, key, cycle_type='phasic', interval_idx=0, frequencies=np.arange(15, 141, 1)):
    """
    Inspect PPC for a specific interval.
    
    Args:
        results: the results dict from run_ppc_dataset
        key: tuple (rat_id, condition, sd, trial_number)
        cycle_type: 'phasic' or 'tonic'
        interval_idx: which interval (0-indexed)
        frequencies: frequency vector
    """
    if key not in results:
        print(f"Key {key} not found in results.")
        print("Available keys:", list(results.keys()))
        return
    
    res = results[key]
    ppc_key = f'{cycle_type}_ppc_per_interval'
    ppc_list = res[ppc_key]
    
    if interval_idx >= len(ppc_list):
        print(f"Only {len(ppc_list)} {cycle_type} intervals available (requested idx={interval_idx})")
        return
    
    ppc = ppc_list[interval_idx]  # (n_cycles, n_freqs)
    if ppc.size == 0:
        print("No cycles in this interval.")
        return
    
    n_valid = np.sum(~np.isnan(ppc[:, 0]))
    label = f"Rat{key[0]} | {key[1]} | SD{key[2]} | Trial{key[3]} | {cycle_type} interval {interval_idx}"
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Mean PPC spectrum for this interval
    ax = axes[0]
    mean_ppc = np.nanmean(ppc, axis=0)
    sem_ppc = np.nanstd(ppc, axis=0) / np.sqrt(n_valid) if n_valid > 1 else np.zeros_like(mean_ppc)
    color = 'crimson' if cycle_type == 'phasic' else 'steelblue'
    ax.plot(frequencies, mean_ppc, color=color)
    ax.fill_between(frequencies, mean_ppc - sem_ppc, mean_ppc + sem_ppc, color=color, alpha=0.2)
    ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PPC')
    ax.set_title(f'Mean PPC ({n_valid} cycles)')
    
    # 2. Per-cycle PPC heatmap
    ax = axes[1]
    im = ax.imshow(ppc, aspect='auto', origin='lower', interpolation='nearest',
                   extent=[frequencies[0], frequencies[-1], 0, ppc.shape[0]],
                   cmap='hot')
    plt.colorbar(im, ax=ax, label='PPC')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Cycle #')
    ax.set_title('Per-Cycle PPC')
    
    # 3. Individual cycle PPC traces (up to 20)
    ax = axes[2]
    n_show = min(20, ppc.shape[0])
    for ci in range(n_show):
        if not np.all(np.isnan(ppc[ci])):
            ax.plot(frequencies, ppc[ci], alpha=0.3, linewidth=0.7, color=color)
    ax.plot(frequencies, mean_ppc, color='black', linewidth=2, label='Mean')
    ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PPC')
    ax.set_title(f'Individual Cycles (showing {n_show})')
    ax.legend()
    
    plt.suptitle(label, fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

print("Inspection functions defined.")

In [ ]:
# List all available intervals (filter by rat_id if desired)
list_available_intervals(results, metadata, rat_id=None)  # set rat_id=3 to filter

In [ ]:
# --- Inspect a specific interval ---
# Example: look at phasic interval 0 from the first available trial
# Change key, cycle_type, and interval_idx as needed

# Pick first available key as example
example_key = metadata[0]['key']
print(f"Inspecting: {example_key}")

inspect_interval(results, example_key, cycle_type='phasic', interval_idx=0, frequencies=frequencies)

In [ ]:
# --- Inspect tonic from same trial ---
inspect_interval(results, example_key, cycle_type='tonic', interval_idx=0, frequencies=frequencies)

### positive rats

In [ ]:
# ---- CHOOSE RATS HERE ----
# Set to None for all rats, or a list like [1, 3, 4] for specific rats
selected_rats = [2, 4, 7, 8]  # e.g. [1, 3] or None for all

results, metadata = run_ppc_dataset(rat_ids=selected_rats, frequencies=frequencies)

# Zhang-Style Inter-Cycle PPC (Vinck et al., 2012)

**Key difference from the previous PPC sections**: Zhang et al. (2019) compute PPC **across theta cycles**, not within a single cycle.

For each frequency `f` and theta phase bin `θ`:
1. Extract the unit-vector cross-spectrum `W_k(f,θ)` for each theta cycle `k`
2. Compute PPC across all pairs of cycles:

```
PPC(f,θ) = [Σ_k Σ_j W_k · conj(W_j) - N] / [N(N-1)]
```

This measures: "Is the HPC-PFC gamma phase difference at frequency f and theta phase θ **consistent across different theta cycles**?"

Pipeline:
- **A**: Compute wavelet cross-spectrum, bin by theta phase → per-cycle FPC vectors
- **B**: Compute Zhang PPC on pooled phasic/tonic cycles  
- **C**: Dataset-wide iteration
- **D**: Visualization (Zhang Figure 4 style)
- **E**: Statistics

## Section A: Core Functions — Cross-Spectrum, Phase Binning, Zhang PPC

In [ ]:
def compute_cross_spectrum_unit_vectors(hpc_cwt, pfc_cwt):
    """
    Compute the unit-vector cross-spectrum between HPC and PFC.
    
    Cross-spectrum: S_xy(t,f) = X(t,f) * conj(Y(t,f))
    Unit vector:    W(t,f) = S_xy / |S_xy|
    
    This gives the instantaneous phase difference as a unit complex number.
    
    Args:
        hpc_cwt: complex wavelet coefficients (n_freqs, n_timepoints)
        pfc_cwt: complex wavelet coefficients (n_freqs, n_timepoints)
    Returns:
        W: unit-vector cross-spectrum (n_freqs, n_timepoints), complex
    """
    cross_spec = hpc_cwt * np.conj(pfc_cwt)
    magnitude = np.abs(cross_spec)
    # Avoid division by zero
    magnitude[magnitude == 0] = 1e-16
    W = cross_spec / magnitude
    return W


def bin_cross_spectrum_by_theta_phase(W, cycle_inds, bin_count=20):
    """
    Bin unit-vector cross-spectrum into theta phase bins for each cycle.
    
    Following Zhang et al.: each theta cycle is divided into `bin_count` 
    equally-spaced phase bins. The cross-spectrum at each time point is 
    assigned to the corresponding bin.
    
    Args:
        W: unit-vector cross-spectrum (n_freqs, n_timepoints)
        cycle_inds: Nx2 array of [start, end] sample indices
        bin_count: number of theta phase bins (Zhang uses 20)
    Returns:
        fpc_matrices: list of N arrays, each (n_freqs, bin_count) complex.
                      FPC = Frequency-Phase Cross-spectrogram for each cycle.
                      Each bin contains the mean unit-vector cross-spectrum.
    """
    n_freqs = W.shape[0]
    n_cycles = cycle_inds.shape[0]
    fpc_matrices = []
    
    for i in range(n_cycles):
        start, end = cycle_inds[i]
        n_samples = end - start
        
        if n_samples < bin_count:
            # Cycle too short — fill with NaN
            fpc = np.full((n_freqs, bin_count), np.nan, dtype=complex)
            fpc_matrices.append(fpc)
            continue
        
        # Assign each time point to a phase bin (evenly across cycle)
        bin_assignments = np.floor(np.linspace(0, bin_count, n_samples, endpoint=False)).astype(int)
        bin_assignments = np.clip(bin_assignments, 0, bin_count - 1)
        
        fpc = np.full((n_freqs, bin_count), np.nan, dtype=complex)
        for b in range(bin_count):
            mask = bin_assignments == b
            if np.any(mask):
                # Mean unit-vector cross-spectrum in this phase bin
                fpc[:, b] = np.mean(W[:, start:end][:, mask], axis=1)
        
        fpc_matrices.append(fpc)
    
    return fpc_matrices


def compute_zhang_ppc(fpc_list, min_cycles=10):
    """
    Compute Zhang-style inter-cycle PPC (Vinck et al., 2012).
    
    PPC(f, θ) = [Σ_k Σ_j Re(W_k · conj(W_j)) - N] / [N(N-1)]
    
    where W_k(f,θ) is the unit-vector cross-spectrum for cycle k at 
    frequency f and theta phase θ.
    
    Efficient computation:
        PPC = (N * |mean(W)|^2 - 1) / (N - 1)
    
    Args:
        fpc_list: list of FPC matrices (n_freqs, bin_count), complex.
                  Each element is one theta cycle's FPC.
        min_cycles: minimum number of valid cycles required
    Returns:
        ppc_matrix: (n_freqs, bin_count) PPC values
        n_valid: (n_freqs, bin_count) number of valid cycles per bin
    """
    if len(fpc_list) == 0:
        return None, None
    
    n_freqs, bin_count = fpc_list[0].shape
    
    # Stack all FPC matrices: (n_cycles, n_freqs, bin_count)
    fpc_stack = np.array(fpc_list)  # (N, n_freqs, bin_count)
    
    # For each (f, θ), count valid (non-NaN) cycles
    valid_mask = ~np.isnan(fpc_stack)  # NaN check on real part suffices for complex
    # Complex NaN: both real and imag are NaN
    valid_mask = np.isfinite(fpc_stack.real) & np.isfinite(fpc_stack.imag)
    n_valid = np.sum(valid_mask, axis=0)  # (n_freqs, bin_count)
    
    # Replace NaN with 0 for summation
    fpc_clean = np.where(valid_mask, fpc_stack, 0.0 + 0.0j)
    
    # Mean unit vector across cycles
    sum_W = np.sum(fpc_clean, axis=0)  # (n_freqs, bin_count)
    
    # PPC = (N * |mean(W)|^2 - 1) / (N - 1)
    # mean(W) = sum_W / N, so |mean(W)|^2 = |sum_W|^2 / N^2
    # PPC = (|sum_W|^2 / N - 1) / (N - 1)  = (|sum_W|^2 - N) / (N * (N - 1))
    ppc_matrix = np.full((n_freqs, bin_count), np.nan)
    
    valid = n_valid >= min_cycles
    N = n_valid[valid]
    ppc_matrix[valid] = (np.abs(sum_W[valid])**2 - N) / (N * (N - 1))
    
    return ppc_matrix, n_valid


def compute_zhang_ppc_spectrum(fpc_list, phase_range=None, min_cycles=10):
    """
    Compute Zhang PPC as a function of frequency, optionally averaged 
    over a specific theta phase range.
    
    If phase_range is None, averages over all phase bins.
    
    Args:
        fpc_list: list of FPC matrices (n_freqs, bin_count), complex
        phase_range: tuple (start_bin, end_bin) to average over, or None for all
        min_cycles: minimum valid cycles
    Returns:
        ppc_spectrum: (n_freqs,) array
        ppc_matrix: full (n_freqs, bin_count) matrix
        n_valid: (n_freqs, bin_count) counts
    """
    ppc_matrix, n_valid = compute_zhang_ppc(fpc_list, min_cycles=min_cycles)
    
    if ppc_matrix is None:
        return None, None, None
    
    if phase_range is not None:
        start_b, end_b = phase_range
        ppc_spectrum = np.nanmean(ppc_matrix[:, start_b:end_b], axis=1)
    else:
        ppc_spectrum = np.nanmean(ppc_matrix, axis=1)
    
    return ppc_spectrum, ppc_matrix, n_valid

print("Zhang PPC core functions defined.")
print("  - compute_cross_spectrum_unit_vectors()")
print("  - bin_cross_spectrum_by_theta_phase()")
print("  - compute_zhang_ppc()")
print("  - compute_zhang_ppc_spectrum()")

## Section B: Single-Recording Demo (Rat1 Post-Trial5)

Fully self-contained: loads data, extracts intervals, computes IMFs, wavelets, cross-spectra, and Zhang PPC.

In [ ]:
# === Fully self-contained single-recording Zhang PPC ===

# --- Parameters ---
path_to_hpc = '/Users/amir/Desktop/for Abdel/RGS/DatabyCondition/HomeCageHC/OS_Ephys_RGS14_Rat1_57986_SD4_HC_01_08_2018/2018-08-01_15-49-15_Post-Trial5/HPC_100_CH4_0.continuous.mat'
path_to_pfc = '/Users/amir/Desktop/for Abdel/RGS/DatabyCondition/HomeCageHC/OS_Ephys_RGS14_Rat1_57986_SD4_HC_01_08_2018/2018-08-01_15-49-15_Post-Trial5/PFC_100_CH47_0.continuous.mat'
path_to_hypno = '/Users/amir/Desktop/for Abdel/RGS/DatabyCondition/HomeCageHC/OS_Ephys_RGS14_Rat1_57986_SD4_HC_01_08_2018/2018-08-01_15-49-15_Post-Trial5/2018-08-01_15-49-15_Post-Trial5_concatenated-states.mat'

frequencies = np.arange(15, 141, 1)
fs = 1000
n_cycles_wavelet = 5
bin_count_zhang = 20

# --- Load data ---
print("Loading HPC and PFC data...")
lfpHPC, hypno, _ = get_data(path_to_hpc, path_to_hypno)
lfpPFC, _, _ = get_data(path_to_pfc, path_to_hypno, type='pfc')

# --- Extract phasic/tonic intervals ---
print("Extracting phasic/tonic REM intervals...")
phasic_interval, tonic_interval, lfp_raw = extract_pt_intervals(lfpHPC, hypno, fs=fs)

# --- IMFs ---
print("Computing IMFs...")
phasic_imfs, phasic_freqs, _ = extract_imfs_by_pt_intervals(
    lfp_raw, fs, phasic_interval, config, return_imfs_freqs=True)
tonic_imfs, tonic_freqs, _ = extract_imfs_by_pt_intervals(
    lfp_raw, fs, tonic_interval, config, return_imfs_freqs=True)

# --- Cycle indices and HPC gamma ---
print("Extracting theta cycles and HPC gamma...")
phasic_cycle_inds, phasic_hpc_gamma = extract_cycle_indices_and_gamma(phasic_imfs, phasic_freqs)
tonic_cycle_inds, tonic_hpc_gamma = extract_cycle_indices_and_gamma(tonic_imfs, tonic_freqs)

# --- PFC: extract and bandpass filter ---
print("Filtering PFC to gamma...")
pfc_phasic = extract_lfp_by_pt_intervals(lfpPFC, fs, phasic_interval)
pfc_tonic = extract_lfp_by_pt_intervals(lfpPFC, fs, tonic_interval)
pfc_phasic_filt = [bandpass_filter(sig, fs=fs, low=15, high=200, order=4) for sig in pfc_phasic]
pfc_tonic_filt = [bandpass_filter(sig, fs=fs, low=15, high=200, order=4) for sig in pfc_tonic]

# --- Complex wavelets ---
print("Computing complex wavelets (HPC)...")
phasic_hpc_cwt = compute_complex_wavelets(phasic_hpc_gamma, frequencies, fs, n_cycles=n_cycles_wavelet)
tonic_hpc_cwt = compute_complex_wavelets(tonic_hpc_gamma, frequencies, fs, n_cycles=n_cycles_wavelet)
print("Computing complex wavelets (PFC)...")
phasic_pfc_cwt = compute_complex_wavelets(pfc_phasic_filt, frequencies, fs, n_cycles=n_cycles_wavelet)
tonic_pfc_cwt = compute_complex_wavelets(pfc_tonic_filt, frequencies, fs, n_cycles=n_cycles_wavelet)

# --- Cross-spectrum → FPC matrices per cycle ---
print("Computing cross-spectrum FPC matrices...")
phasic_fpc_all = []
for hcwt, pcwt, cinds in zip(phasic_hpc_cwt, phasic_pfc_cwt, phasic_cycle_inds):
    if len(cinds) == 0:
        continue
    W = compute_cross_spectrum_unit_vectors(hcwt, pcwt)
    fpc_list = bin_cross_spectrum_by_theta_phase(W, cinds, bin_count=bin_count_zhang)
    phasic_fpc_all.extend(fpc_list)

tonic_fpc_all = []
for hcwt, pcwt, cinds in zip(tonic_hpc_cwt, tonic_pfc_cwt, tonic_cycle_inds):
    if len(cinds) == 0:
        continue
    W = compute_cross_spectrum_unit_vectors(hcwt, pcwt)
    fpc_list = bin_cross_spectrum_by_theta_phase(W, cinds, bin_count=bin_count_zhang)
    tonic_fpc_all.extend(fpc_list)

print(f"Phasic FPC matrices: {len(phasic_fpc_all)} cycles")
print(f"Tonic FPC matrices:  {len(tonic_fpc_all)} cycles")

# --- Compute Zhang PPC ---
print("Computing Zhang PPC...")
phasic_zhang_ppc, phasic_zhang_nvalid = compute_zhang_ppc(phasic_fpc_all, min_cycles=5)
tonic_zhang_ppc, tonic_zhang_nvalid = compute_zhang_ppc(tonic_fpc_all, min_cycles=5)

print(f"\nPhasic PPC shape: {phasic_zhang_ppc.shape if phasic_zhang_ppc is not None else 'None'}")
print(f"Tonic PPC shape: {tonic_zhang_ppc.shape if tonic_zhang_ppc is not None else 'None'}")
if phasic_zhang_ppc is not None:
    print(f"Phasic PPC range: [{np.nanmin(phasic_zhang_ppc):.4f}, {np.nanmax(phasic_zhang_ppc):.4f}]")
if tonic_zhang_ppc is not None:
    print(f"Tonic PPC range: [{np.nanmin(tonic_zhang_ppc):.4f}, {np.nanmax(tonic_zhang_ppc):.4f}]")
print("Done.")

In [ ]:
# --- Visualize single-recording Zhang PPC ---
phase_bins_deg = np.linspace(-180, 180, bin_count_zhang + 1)
phase_centers_deg = (phase_bins_deg[:-1] + phase_bins_deg[1:]) / 2

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Shared colorbar limits
vmin_shared, vmax_shared = None, None
for ppc in [phasic_zhang_ppc, tonic_zhang_ppc]:
    if ppc is not None:
        fmin, fmax = np.nanmin(ppc), np.nanmax(ppc)
        vmin_shared = fmin if vmin_shared is None else min(vmin_shared, fmin)
        vmax_shared = fmax if vmax_shared is None else max(vmax_shared, fmax)

for ax, ppc, nv, label in zip(axes[:2], 
                               [phasic_zhang_ppc, tonic_zhang_ppc],
                               [phasic_zhang_nvalid, tonic_zhang_nvalid],
                               ['Phasic', 'Tonic']):
    if ppc is None:
        ax.set_title(f'{label}: insufficient cycles')
        continue
    
    im = ax.imshow(ppc, aspect='auto', origin='lower', interpolation='bilinear',
                   extent=[-180, 180, frequencies[0], frequencies[-1]],
                   cmap='hot', vmin=vmin_shared, vmax=vmax_shared)
    
    # Mode frequency
    ppc_per_freq = np.nanmax(ppc, axis=1)
    mode_idx = int(np.nanargmax(ppc_per_freq))
    mode_freq = float(frequencies[mode_idx])
    ax.axhline(mode_freq, color='cyan', linestyle='--', linewidth=1.5,
               label=f'Mode {mode_freq:.1f} Hz')
    
    plt.colorbar(im, ax=ax, label='PPC')
    ax.set_xlabel('Theta Phase (deg)')
    ax.set_ylabel('Frequency (Hz)')
    n_cycles_used = int(np.nanmean(nv)) if nv is not None else 0
    ax.set_title(f'{label} REM | Zhang PPC (n~{n_cycles_used} cycles)')
    ax.legend(loc='upper right', frameon=False)

# Difference
if phasic_zhang_ppc is not None and tonic_zhang_ppc is not None:
    ax = axes[2]
    diff = phasic_zhang_ppc - tonic_zhang_ppc
    vmax_diff = np.nanmax(np.abs(diff))
    im = ax.imshow(diff, aspect='auto', origin='lower', interpolation='bilinear',
                   extent=[-180, 180, frequencies[0], frequencies[-1]],
                   cmap='RdBu_r', vmin=-vmax_diff, vmax=vmax_diff)
    plt.colorbar(im, ax=ax, label='PPC difference')
    ax.set_xlabel('Theta Phase (deg)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title('Phasic - Tonic')

plt.suptitle('Zhang-Style Inter-Cycle PPC: HPC-PFC (Single Recording)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- PPC spectrum (averaged across all theta phases) ---
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

if phasic_zhang_ppc is not None:
    phasic_spectrum = np.nanmean(phasic_zhang_ppc, axis=1)
    ax.plot(frequencies, phasic_spectrum, color='crimson', linewidth=2, label='Phasic')

if tonic_zhang_ppc is not None:
    tonic_spectrum = np.nanmean(tonic_zhang_ppc, axis=1)
    ax.plot(frequencies, tonic_spectrum, color='steelblue', linewidth=2, label='Tonic')

ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PPC (Zhang, inter-cycle)')
ax.set_title('Zhang PPC Spectrum: HPC-PFC (Single Recording, avg over all theta phases)')
ax.legend()
plt.tight_layout()
plt.show()

## Section C: Full Dataset — Zhang PPC

Iterate over all rats/conditions/trials. For each trial, extract FPC matrices per cycle. Then pool across trials to compute grand PPC.

In [ ]:
def compute_zhang_fpc_for_trial(lfp_file, pfc_file, state_file, config, 
                                frequencies, fs=1000, n_cycles_wavelet=5, 
                                bin_count=20):
    """
    Full Zhang PPC pipeline for a single trial.
    
    Returns per-cycle FPC matrices AND per-cycle wavelet snippets 
    (needed for surrogate Z-scoring).
    """
    try:
        lfpHPC, hypno, _ = get_data(lfp_file, state_file)
        lfpPFC, _, _ = get_data(pfc_file, state_file, type='pfc')
    except Exception as e:
        print(f"  Failed to load data: {e}")
        return None
    
    try:
        phasic_interval, tonic_interval, lfp_raw = extract_pt_intervals(lfpHPC, hypno, fs=fs)
    except ValueError:
        print(f"  No REM sleep found.")
        return None
    
    if not phasic_interval or not tonic_interval:
        print(f"  No phasic or tonic intervals.")
        return None
    
    if hasattr(phasic_interval, '__len__') and len(phasic_interval) == 0:
        return None
    if hasattr(tonic_interval, '__len__') and len(tonic_interval) == 0:
        return None
    
    # Extract IMFs
    try:
        phasic_imfs, phasic_freqs, _ = extract_imfs_by_pt_intervals(
            lfp_raw, fs, phasic_interval, config, return_imfs_freqs=True)
        tonic_imfs, tonic_freqs, _ = extract_imfs_by_pt_intervals(
            lfp_raw, fs, tonic_interval, config, return_imfs_freqs=True)
    except Exception as e:
        print(f"  IMF extraction failed: {e}")
        return None
    
    # Extract cycle indices and HPC gamma
    phasic_cycle_inds, phasic_hpc_gamma = extract_cycle_indices_and_gamma(phasic_imfs, phasic_freqs)
    tonic_cycle_inds, tonic_hpc_gamma = extract_cycle_indices_and_gamma(tonic_imfs, tonic_freqs)
    
    n_phasic = sum(len(c) for c in phasic_cycle_inds)
    n_tonic = sum(len(c) for c in tonic_cycle_inds)
    if n_phasic == 0 and n_tonic == 0:
        print(f"  No valid theta cycles.")
        return None
    
    # PFC: extract and bandpass filter
    pfc_phasic = extract_lfp_by_pt_intervals(lfpPFC, fs, phasic_interval)
    pfc_tonic = extract_lfp_by_pt_intervals(lfpPFC, fs, tonic_interval)
    pfc_phasic_filt = [bandpass_filter(sig, fs=fs, low=15, high=200, order=4) for sig in pfc_phasic]
    pfc_tonic_filt = [bandpass_filter(sig, fs=fs, low=15, high=200, order=4) for sig in pfc_tonic]
    
    # Complex wavelets
    phasic_hpc_cwt = compute_complex_wavelets(phasic_hpc_gamma, frequencies, fs, n_cycles=n_cycles_wavelet)
    tonic_hpc_cwt = compute_complex_wavelets(tonic_hpc_gamma, frequencies, fs, n_cycles=n_cycles_wavelet)
    phasic_pfc_cwt = compute_complex_wavelets(pfc_phasic_filt, frequencies, fs, n_cycles=n_cycles_wavelet)
    tonic_pfc_cwt = compute_complex_wavelets(pfc_tonic_filt, frequencies, fs, n_cycles=n_cycles_wavelet)
    
    # Cross-spectrum → FPC matrices per cycle
    # ALSO store per-cycle HPC and PFC phase snippets for surrogate Z-scoring
    phasic_fpc_all = []
    phasic_hpc_phases = []  # per-cycle: list of (n_freqs, bin_count) complex (binned HPC phase)
    phasic_pfc_phases = []  # per-cycle: list of (n_freqs, bin_count) complex (binned PFC phase)
    
    for hcwt, pcwt, cinds in zip(phasic_hpc_cwt, phasic_pfc_cwt, phasic_cycle_inds):
        if len(cinds) == 0:
            continue
        W = compute_cross_spectrum_unit_vectors(hcwt, pcwt)
        fpc_list = bin_cross_spectrum_by_theta_phase(W, cinds, bin_count=bin_count)
        phasic_fpc_all.extend(fpc_list)
        
        # Store per-cycle binned unit-vector phases for HPC and PFC separately
        n_freqs = hcwt.shape[0]
        hpc_unit = hcwt / (np.abs(hcwt) + 1e-16)  # unit vector
        pfc_unit = pcwt / (np.abs(pcwt) + 1e-16)
        
        for i in range(cinds.shape[0]):
            start, end = cinds[i]
            n_samples = end - start
            if n_samples < bin_count:
                phasic_hpc_phases.append(np.full((n_freqs, bin_count), np.nan, dtype=complex))
                phasic_pfc_phases.append(np.full((n_freqs, bin_count), np.nan, dtype=complex))
                continue
            bin_assignments = np.floor(np.linspace(0, bin_count, n_samples, endpoint=False)).astype(int)
            bin_assignments = np.clip(bin_assignments, 0, bin_count - 1)
            hpc_binned = np.full((n_freqs, bin_count), np.nan, dtype=complex)
            pfc_binned = np.full((n_freqs, bin_count), np.nan, dtype=complex)
            for b in range(bin_count):
                mask = bin_assignments == b
                if np.any(mask):
                    hpc_binned[:, b] = np.mean(hpc_unit[:, start:end][:, mask], axis=1)
                    pfc_binned[:, b] = np.mean(pfc_unit[:, start:end][:, mask], axis=1)
            phasic_hpc_phases.append(hpc_binned)
            phasic_pfc_phases.append(pfc_binned)
    
    tonic_fpc_all = []
    tonic_hpc_phases = []
    tonic_pfc_phases = []
    
    for hcwt, pcwt, cinds in zip(tonic_hpc_cwt, tonic_pfc_cwt, tonic_cycle_inds):
        if len(cinds) == 0:
            continue
        W = compute_cross_spectrum_unit_vectors(hcwt, pcwt)
        fpc_list = bin_cross_spectrum_by_theta_phase(W, cinds, bin_count=bin_count)
        tonic_fpc_all.extend(fpc_list)
        
        n_freqs = hcwt.shape[0]
        hpc_unit = hcwt / (np.abs(hcwt) + 1e-16)
        pfc_unit = pcwt / (np.abs(pcwt) + 1e-16)
        
        for i in range(cinds.shape[0]):
            start, end = cinds[i]
            n_samples = end - start
            if n_samples < bin_count:
                tonic_hpc_phases.append(np.full((n_freqs, bin_count), np.nan, dtype=complex))
                tonic_pfc_phases.append(np.full((n_freqs, bin_count), np.nan, dtype=complex))
                continue
            bin_assignments = np.floor(np.linspace(0, bin_count, n_samples, endpoint=False)).astype(int)
            bin_assignments = np.clip(bin_assignments, 0, bin_count - 1)
            hpc_binned = np.full((n_freqs, bin_count), np.nan, dtype=complex)
            pfc_binned = np.full((n_freqs, bin_count), np.nan, dtype=complex)
            for b in range(bin_count):
                mask = bin_assignments == b
                if np.any(mask):
                    hpc_binned[:, b] = np.mean(hpc_unit[:, start:end][:, mask], axis=1)
                    pfc_binned[:, b] = np.mean(pfc_unit[:, start:end][:, mask], axis=1)
            tonic_hpc_phases.append(hpc_binned)
            tonic_pfc_phases.append(pfc_binned)
    
    return {
        'phasic_fpc': phasic_fpc_all,
        'tonic_fpc': tonic_fpc_all,
        'n_phasic_cycles': len(phasic_fpc_all),
        'n_tonic_cycles': len(tonic_fpc_all),
        # Per-cycle binned phases for surrogate Z-scoring
        'phasic_hpc_phases': phasic_hpc_phases,
        'phasic_pfc_phases': phasic_pfc_phases,
        'tonic_hpc_phases': tonic_hpc_phases,
        'tonic_pfc_phases': tonic_pfc_phases,
    }

print("compute_zhang_fpc_for_trial defined (with surrogate support).")

In [ ]:
def run_zhang_ppc_dataset(rat_ids=None, frequencies=np.arange(15, 141, 1), 
                          fs=1000, n_cycles_wavelet=5, bin_count=20):
    """
    Run Zhang-style PPC across the full dataset.
    
    Collects per-cycle FPC matrices and per-cycle phase snippets from all trials,
    then computes PPC by pooling across cycles (grand, per-rat, per-trial levels).
    Also stores per-cycle HPC/PFC phases for surrogate Z-scoring.
    """
    base_path = '/Users/amir/Desktop/for Abdel/RGS/DatabyCondition'
    cfg = config
    
    preferred_conditions = ["HomeCageHC", "RandomCon", "OverlappingOR", "StableCondOD"]
    conditions = [c for c in preferred_conditions if os.path.isdir(os.path.join(base_path, c))]
    folder_re = re.compile(r'^OS_Ephys_RGS14_Rat(\d+)_\d+_SD(\d+)_([\w-]+)_([\d_-]+)$')
    
    # Storage: per-rat
    rat_phasic_fpc = {}
    rat_tonic_fpc = {}
    rat_phasic_hpc_phases = {}
    rat_phasic_pfc_phases = {}
    rat_tonic_hpc_phases = {}
    rat_tonic_pfc_phases = {}
    
    trial_results = {}
    zhang_metadata = []
    
    for condition_folder in conditions:
        condition_path = os.path.join(base_path, condition_folder)
        
        for rat_folder in sorted(os.listdir(condition_path)):
            full_path = os.path.join(condition_path, rat_folder)
            if not os.path.isdir(full_path):
                continue
            m = folder_re.match(rat_folder)
            if not m:
                continue
            
            rat_id = int(m.group(1))
            if rat_ids is not None and rat_id not in rat_ids:
                continue
            
            sd_number = m.group(2)
            condition = m.group(3)
            
            post_trial_folders = sorted([
                f for f in os.listdir(full_path)
                if os.path.isdir(os.path.join(full_path, f)) and re.search(r'Post[-_]Trial\d+', f)
            ])
            
            for pt_folder in post_trial_folders:
                trial_path = os.path.join(full_path, pt_folder)
                trial_match = re.search(r'Post[-_]Trial(\d+)', pt_folder)
                if not trial_match:
                    continue
                trial_number = int(trial_match.group(1))
                
                hpc_file, pfc_file, state_file = None, None, None
                for fname in os.listdir(trial_path):
                    low = fname.lower()
                    if 'hpc_100' in low and low.endswith('.mat'):
                        hpc_file = os.path.join(trial_path, fname)
                    elif 'pfc_100' in low and low.endswith('.mat'):
                        pfc_file = os.path.join(trial_path, fname)
                    elif 'states' in low and low.endswith('.mat'):
                        state_file = os.path.join(trial_path, fname)
                
                if not hpc_file or not pfc_file or not state_file:
                    continue
                
                key = (rat_id, condition, int(sd_number), trial_number)
                print(f"\n--- Rat{rat_id} | {condition} | SD{sd_number} | Trial{trial_number} ---")
                
                result = compute_zhang_fpc_for_trial(
                    hpc_file, pfc_file, state_file, cfg, frequencies, 
                    fs, n_cycles_wavelet, bin_count)
                
                if result is not None:
                    trial_results[key] = result
                    
                    # Accumulate per-rat: FPC and phases
                    rat_phasic_fpc.setdefault(rat_id, []).extend(result['phasic_fpc'])
                    rat_tonic_fpc.setdefault(rat_id, []).extend(result['tonic_fpc'])
                    rat_phasic_hpc_phases.setdefault(rat_id, []).extend(result['phasic_hpc_phases'])
                    rat_phasic_pfc_phases.setdefault(rat_id, []).extend(result['phasic_pfc_phases'])
                    rat_tonic_hpc_phases.setdefault(rat_id, []).extend(result['tonic_hpc_phases'])
                    rat_tonic_pfc_phases.setdefault(rat_id, []).extend(result['tonic_pfc_phases'])
                    
                    zhang_metadata.append({
                        'rat_id': rat_id, 'condition': condition,
                        'sd': int(sd_number), 'trial': trial_number,
                        'key': key,
                        'n_phasic_cycles': result['n_phasic_cycles'],
                        'n_tonic_cycles': result['n_tonic_cycles'],
                    })
                    print(f"  OK: {result['n_phasic_cycles']} phasic, {result['n_tonic_cycles']} tonic cycles")
                else:
                    print(f"  SKIPPED")
    
    # --- Pool all cycles across dataset ---
    all_phasic_fpc, all_tonic_fpc = [], []
    all_phasic_hpc_phases, all_phasic_pfc_phases = [], []
    all_tonic_hpc_phases, all_tonic_pfc_phases = [], []
    
    for rid in rat_phasic_fpc:
        all_phasic_fpc.extend(rat_phasic_fpc[rid])
        all_phasic_hpc_phases.extend(rat_phasic_hpc_phases[rid])
        all_phasic_pfc_phases.extend(rat_phasic_pfc_phases[rid])
    for rid in rat_tonic_fpc:
        all_tonic_fpc.extend(rat_tonic_fpc[rid])
        all_tonic_hpc_phases.extend(rat_tonic_hpc_phases[rid])
        all_tonic_pfc_phases.extend(rat_tonic_pfc_phases[rid])
    
    total_phasic = len(all_phasic_fpc)
    total_tonic = len(all_tonic_fpc)
    
    print(f"\n{'='*60}")
    print(f"Total recordings: {len(trial_results)}")
    print(f"Total phasic cycles: {total_phasic}")
    print(f"Total tonic cycles: {total_tonic}")
    print(f"Rats: {sorted(rat_phasic_fpc.keys())}")
    
    # --- Compute PPC at different levels ---
    print("\nComputing grand PPC...")
    grand_phasic_ppc, grand_phasic_nv = compute_zhang_ppc(all_phasic_fpc, min_cycles=10)
    grand_tonic_ppc, grand_tonic_nv = compute_zhang_ppc(all_tonic_fpc, min_cycles=10)
    
    print("Computing per-rat PPC...")
    per_rat_phasic_ppc = {}
    per_rat_tonic_ppc = {}
    for rid in sorted(set(rat_phasic_fpc.keys()) | set(rat_tonic_fpc.keys())):
        if rid in rat_phasic_fpc and len(rat_phasic_fpc[rid]) >= 10:
            per_rat_phasic_ppc[rid], _ = compute_zhang_ppc(rat_phasic_fpc[rid], min_cycles=10)
        if rid in rat_tonic_fpc and len(rat_tonic_fpc[rid]) >= 10:
            per_rat_tonic_ppc[rid], _ = compute_zhang_ppc(rat_tonic_fpc[rid], min_cycles=10)
        n_p = len(rat_phasic_fpc.get(rid, []))
        n_t = len(rat_tonic_fpc.get(rid, []))
        print(f"  Rat{rid}: {n_p} phasic, {n_t} tonic cycles")
    
    print("Computing per-trial PPC...")
    per_trial_phasic_ppc = {}
    per_trial_tonic_ppc = {}
    for key, res in trial_results.items():
        if len(res['phasic_fpc']) >= 10:
            per_trial_phasic_ppc[key], _ = compute_zhang_ppc(res['phasic_fpc'], min_cycles=5)
        if len(res['tonic_fpc']) >= 10:
            per_trial_tonic_ppc[key], _ = compute_zhang_ppc(res['tonic_fpc'], min_cycles=5)
    
    return {
        'grand_phasic_ppc': grand_phasic_ppc,
        'grand_tonic_ppc': grand_tonic_ppc,
        'grand_phasic_nvalid': grand_phasic_nv,
        'grand_tonic_nvalid': grand_tonic_nv,
        'per_rat_phasic_ppc': per_rat_phasic_ppc,
        'per_rat_tonic_ppc': per_rat_tonic_ppc,
        'per_trial_phasic_ppc': per_trial_phasic_ppc,
        'per_trial_tonic_ppc': per_trial_tonic_ppc,
        'total_phasic_cycles': total_phasic,
        'total_tonic_cycles': total_tonic,
        'rat_ids': sorted(set(rat_phasic_fpc.keys()) | set(rat_tonic_fpc.keys())),
        # Phase data for surrogate Z-scoring
        'all_phasic_hpc_phases': all_phasic_hpc_phases,
        'all_phasic_pfc_phases': all_phasic_pfc_phases,
        'all_tonic_hpc_phases': all_tonic_hpc_phases,
        'all_tonic_pfc_phases': all_tonic_pfc_phases,
        'rat_phasic_hpc_phases': rat_phasic_hpc_phases,
        'rat_phasic_pfc_phases': rat_phasic_pfc_phases,
        'rat_tonic_hpc_phases': rat_tonic_hpc_phases,
        'rat_tonic_pfc_phases': rat_tonic_pfc_phases,
    }, zhang_metadata

print("run_zhang_ppc_dataset defined (with surrogate support).")

In [ ]:
# ---- CHOOSE RATS HERE ----
zhang_rat_ids = [1, 3, 6, 9]  # None for all rats, or e.g. [1, 3]

zhang_results, zhang_metadata = run_zhang_ppc_dataset(
    rat_ids=zhang_rat_ids, frequencies=frequencies, bin_count=20)

## Section D: Visualization — Zhang PPC

In [ ]:
# --- Plot 1: Grand-average Zhang PPC heatmaps (phasic, tonic, difference) ---
gp = zhang_results['grand_phasic_ppc']
gt = zhang_results['grand_tonic_ppc']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Shared colorbar
vmin_shared, vmax_shared = None, None
for ppc in [gp, gt]:
    if ppc is not None:
        fmin, fmax = np.nanmin(ppc), np.nanmax(ppc)
        vmin_shared = fmin if vmin_shared is None else min(vmin_shared, fmin)
        vmax_shared = fmax if vmax_shared is None else max(vmax_shared, fmax)

for ax, ppc, nv, label in zip(axes[:2], [gp, gt],
                               [zhang_results['grand_phasic_nvalid'], zhang_results['grand_tonic_nvalid']],
                               ['Phasic', 'Tonic']):
    if ppc is None:
        ax.set_title(f'{label}: insufficient cycles')
        continue
    
    im = ax.imshow(ppc, aspect='auto', origin='lower', interpolation='bilinear',
                   extent=[-180, 180, frequencies[0], frequencies[-1]],
                   cmap='hot', vmin=vmin_shared, vmax=vmax_shared)
    
    ppc_per_freq = np.nanmax(ppc, axis=1)
    mode_idx = int(np.nanargmax(ppc_per_freq))
    mode_freq = float(frequencies[mode_idx])
    ax.axhline(mode_freq, color='cyan', linestyle='--', linewidth=1.5,
               label=f'Mode {mode_freq:.1f} Hz')
    
    plt.colorbar(im, ax=ax, label='PPC')
    ax.set_xlabel('Theta Phase (deg)')
    ax.set_ylabel('Frequency (Hz)')
    n_cyc = zhang_results[f'total_{label.lower()}_cycles']
    ax.set_title(f'{label} REM | Grand Zhang PPC (N={n_cyc})')
    ax.legend(loc='upper right', frameon=False)

# Difference
if gp is not None and gt is not None:
    ax = axes[2]
    diff = gp - gt
    vmax_diff = np.nanmax(np.abs(diff))
    im = ax.imshow(diff, aspect='auto', origin='lower', interpolation='bilinear',
                   extent=[-180, 180, frequencies[0], frequencies[-1]],
                   cmap='RdBu_r', vmin=-vmax_diff, vmax=vmax_diff)
    plt.colorbar(im, ax=ax, label='PPC difference')
    ax.set_xlabel('Theta Phase (deg)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title('Phasic - Tonic (Grand)')

plt.suptitle('Zhang-Style Inter-Cycle PPC: HPC-PFC Gamma (All Rats)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 2: Grand PPC spectrum (averaged over all theta phases) ---
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

gp = zhang_results['grand_phasic_ppc']
gt = zhang_results['grand_tonic_ppc']

if gp is not None:
    phasic_spec = np.nanmean(gp, axis=1)
    ax.plot(frequencies, phasic_spec, color='crimson', linewidth=2, 
            label=f'Phasic (N={zhang_results["total_phasic_cycles"]})')

if gt is not None:
    tonic_spec = np.nanmean(gt, axis=1)
    ax.plot(frequencies, tonic_spec, color='steelblue', linewidth=2,
            label=f'Tonic (N={zhang_results["total_tonic_cycles"]})')

ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PPC (Zhang, inter-cycle)')
ax.set_title('Grand Zhang PPC Spectrum: HPC-PFC (avg over all theta phases)')
ax.legend()
plt.tight_layout()
plt.show()

### 1/f Normalization

Fit `PPC(f) = a * f^(-b) + c` to the **pooled** PPC spectrum (phasic + tonic combined), then divide both phasic and tonic PPC matrices by this fitted curve. This removes the shared 1/f bias while preserving real differences between conditions and across theta phases.

In [ ]:
from scipy.optimize import curve_fit

def one_over_f_model(f, a, b, c):
    """1/f model: PPC(f) = a * f^(-b) + c"""
    return a * np.power(f, -b) + c

def fit_one_over_f(frequencies, ppc_spectrum):
    """
    Fit 1/f model to a PPC spectrum.
    Returns fitted curve and parameters.
    """
    # Initial guesses
    p0 = [ppc_spectrum[0] * frequencies[0], 1.0, 0.0]
    bounds = ([0, 0, -np.inf], [np.inf, 5, np.inf])
    
    # Only fit on non-NaN values
    valid = np.isfinite(ppc_spectrum)
    try:
        popt, _ = curve_fit(one_over_f_model, frequencies[valid], ppc_spectrum[valid], 
                            p0=p0, bounds=bounds, maxfev=5000)
        fitted = one_over_f_model(frequencies, *popt)
        return fitted, popt
    except RuntimeError:
        print("  1/f fit failed, using simple 1/f fallback")
        return ppc_spectrum, None

# --- Fit 1/f to pooled (phasic + tonic combined) spectrum ---
gp = zhang_results['grand_phasic_ppc']
gt = zhang_results['grand_tonic_ppc']

# Average both conditions to get a shared baseline
pooled_spec = np.nanmean(gp, axis=1) * 0.5 + np.nanmean(gt, axis=1) * 0.5
one_over_f_fit, fit_params = fit_one_over_f(frequencies, pooled_spec)

if fit_params is not None:
    print(f"1/f fit: a={fit_params[0]:.4f}, b={fit_params[1]:.4f}, c={fit_params[2]:.6f}")
    fit_label = f'1/f fit: a·f^(-{fit_params[1]:.2f})+c'
else:
    fit_label = '1/f fit (fallback)'

# --- Normalize: divide PPC matrices by the 1/f curve (broadcast across phase bins) ---
one_over_f_divisor = one_over_f_fit[:, np.newaxis]
one_over_f_divisor[one_over_f_divisor == 0] = 1e-16

phasic_ppc_1f = gp / one_over_f_divisor if gp is not None else None
tonic_ppc_1f = gt / one_over_f_divisor if gt is not None else None

# --- Quick check ---
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(frequencies, pooled_spec, 'k-', linewidth=2, label='Pooled PPC spectrum')
ax.plot(frequencies, one_over_f_fit, 'r--', linewidth=2, label=fit_label)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PPC')
ax.set_title('1/f Fit to Pooled PPC Spectrum')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- 1/f-normalized PPC heatmaps ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Shared colorbar
vmin_1f, vmax_1f = None, None
for ppc in [phasic_ppc_1f, tonic_ppc_1f]:
    if ppc is not None:
        fmin, fmax = np.nanmin(ppc), np.nanmax(ppc)
        vmin_1f = fmin if vmin_1f is None else min(vmin_1f, fmin)
        vmax_1f = fmax if vmax_1f is None else max(vmax_1f, fmax)

for ax, ppc, label, n_cyc in zip(axes[:2],
                                   [phasic_ppc_1f, tonic_ppc_1f],
                                   ['Phasic', 'Tonic'],
                                   [zhang_results['total_phasic_cycles'],
                                    zhang_results['total_tonic_cycles']]):
    if ppc is None:
        ax.set_title(f'{label}: no data')
        continue
    
    im = ax.imshow(ppc, aspect='auto', origin='lower', interpolation='bilinear',
                   extent=[-180, 180, frequencies[0], frequencies[-1]],
                   cmap='hot', vmin=vmin_1f, vmax=vmax_1f)
    
    ppc_per_freq = np.nanmax(ppc, axis=1)
    mode_idx = int(np.nanargmax(ppc_per_freq))
    mode_freq = float(frequencies[mode_idx])
    ax.axhline(mode_freq, color='cyan', linestyle='--', linewidth=1.5,
               label=f'Mode {mode_freq:.1f} Hz')
    
    plt.colorbar(im, ax=ax, label='PPC / 1/f')
    ax.set_xlabel('Theta Phase (deg)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(f'{label} REM | 1/f-Normalized PPC (N={n_cyc})')
    ax.legend(loc='upper right', frameon=False)

# Difference
if phasic_ppc_1f is not None and tonic_ppc_1f is not None:
    ax = axes[2]
    diff = phasic_ppc_1f - tonic_ppc_1f
    vmax_diff = np.nanmax(np.abs(diff))
    im = ax.imshow(diff, aspect='auto', origin='lower', interpolation='bilinear',
                   extent=[-180, 180, frequencies[0], frequencies[-1]],
                   cmap='RdBu_r', vmin=-vmax_diff, vmax=vmax_diff)
    plt.colorbar(im, ax=ax, label='PPC/1f difference')
    ax.set_xlabel('Theta Phase (deg)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title('Phasic - Tonic (1/f-normalized)')

plt.suptitle('1/f-Normalized PPC: HPC-PFC Gamma', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- 1/f-normalized PPC spectrum ---
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

if phasic_ppc_1f is not None:
    phasic_1f_spec = np.nanmean(phasic_ppc_1f, axis=1)
    ax.plot(frequencies, phasic_1f_spec, color='crimson', linewidth=2,
            label=f'Phasic (N={zhang_results["total_phasic_cycles"]})')

if tonic_ppc_1f is not None:
    tonic_1f_spec = np.nanmean(tonic_ppc_1f, axis=1)
    ax.plot(frequencies, tonic_1f_spec, color='steelblue', linewidth=2,
            label=f'Tonic (N={zhang_results["total_tonic_cycles"]})')

ax.axhline(1.0, color='gray', linestyle=':', alpha=0.5, label='1/f baseline (=1)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PPC / 1/f (normalized)')
ax.set_title('1/f-Normalized PPC Spectrum: HPC-PFC')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 3: Per-rat Zhang PPC heatmaps ---
rat_ids_list = zhang_results['rat_ids']
n_rats = len(rat_ids_list)

if n_rats > 0:
    fig, axes = plt.subplots(2, n_rats, figsize=(5 * n_rats, 10), squeeze=False)
    
    for i, rid in enumerate(rat_ids_list):
        # Row 0: Phasic
        ax = axes[0, i]
        ppc = zhang_results['per_rat_phasic_ppc'].get(rid)
        if ppc is not None:
            im = ax.imshow(ppc, aspect='auto', origin='lower', interpolation='bilinear',
                           extent=[-180, 180, frequencies[0], frequencies[-1]], cmap='hot')
            plt.colorbar(im, ax=ax, label='PPC')
            ppc_per_freq = np.nanmax(ppc, axis=1)
            mode_idx = int(np.nanargmax(ppc_per_freq))
            ax.axhline(frequencies[mode_idx], color='cyan', linestyle='--', linewidth=1,
                       label=f'Mode {frequencies[mode_idx]:.0f} Hz')
            ax.legend(loc='upper right', frameon=False, fontsize=7)
        ax.set_title(f'Rat{rid} Phasic', fontsize=10)
        ax.set_xlabel('Theta Phase (deg)')
        if i == 0:
            ax.set_ylabel('Frequency (Hz)')
        
        # Row 1: Tonic
        ax = axes[1, i]
        ppc = zhang_results['per_rat_tonic_ppc'].get(rid)
        if ppc is not None:
            im = ax.imshow(ppc, aspect='auto', origin='lower', interpolation='bilinear',
                           extent=[-180, 180, frequencies[0], frequencies[-1]], cmap='hot')
            plt.colorbar(im, ax=ax, label='PPC')
            ppc_per_freq = np.nanmax(ppc, axis=1)
            mode_idx = int(np.nanargmax(ppc_per_freq))
            ax.axhline(frequencies[mode_idx], color='cyan', linestyle='--', linewidth=1,
                       label=f'Mode {frequencies[mode_idx]:.0f} Hz')
            ax.legend(loc='upper right', frameon=False, fontsize=7)
        ax.set_title(f'Rat{rid} Tonic', fontsize=10)
        ax.set_xlabel('Theta Phase (deg)')
        if i == 0:
            ax.set_ylabel('Frequency (Hz)')
    
    plt.suptitle('Per-Rat Zhang PPC: HPC-PFC Gamma', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Plot 4: Per-rat PPC spectra (line plots, averaged over theta phases) ---
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

colors_phasic = plt.cm.Reds(np.linspace(0.3, 0.9, n_rats))
colors_tonic = plt.cm.Blues(np.linspace(0.3, 0.9, n_rats))

for i, rid in enumerate(rat_ids_list):
    ppc_p = zhang_results['per_rat_phasic_ppc'].get(rid)
    ppc_t = zhang_results['per_rat_tonic_ppc'].get(rid)
    
    if ppc_p is not None:
        spec_p = np.nanmean(ppc_p, axis=1)
        ax.plot(frequencies, spec_p, color=colors_phasic[i], linewidth=1.5,
                label=f'Rat{rid} Phasic', linestyle='-')
    if ppc_t is not None:
        spec_t = np.nanmean(ppc_t, axis=1)
        ax.plot(frequencies, spec_t, color=colors_tonic[i], linewidth=1.5,
                label=f'Rat{rid} Tonic', linestyle='--')

ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PPC (Zhang, inter-cycle)')
ax.set_title('Per-Rat Zhang PPC Spectra (avg over theta phases)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## Section D.2: Surrogate Z-Scored PPC

Shuffle the PFC cycle labels to break the true HPC-PFC pairing. For each surrogate iteration, pair HPC cycle `k` with PFC cycle `shuffle[k]`, recompute the cross-spectrum, then compute PPC. Repeat 200 times to build a null distribution. Z-score the real PPC against this null.

This removes all frequency-dependent biases (1/f, wavelet smearing) and shows only **above-chance** coupling.

In [ ]:
def compute_surrogate_fpc(hpc_phases, pfc_phases, rng):
    """
    Create one surrogate by shuffling PFC cycle order, then recomputing 
    cross-spectrum FPC matrices.
    
    Args:
        hpc_phases: list of N arrays (n_freqs, bin_count), complex — per-cycle HPC
        pfc_phases: list of N arrays (n_freqs, bin_count), complex — per-cycle PFC
        rng: numpy random generator
    
    Returns:
        surrogate_fpc: list of N FPC matrices (n_freqs, bin_count), complex
    """
    N = len(hpc_phases)
    shuffle_idx = rng.permutation(N)
    
    surrogate_fpc = []
    for i in range(N):
        hpc = hpc_phases[i]
        pfc = pfc_phases[shuffle_idx[i]]
        
        # Cross-spectrum of shuffled pair: HPC_k * conj(PFC_shuffle[k])
        cross = hpc * np.conj(pfc)
        # Normalize to unit vector
        mag = np.abs(cross)
        mag[mag == 0] = 1e-16
        fpc = cross / mag
        
        # Preserve NaN structure
        nan_mask = np.isnan(hpc) | np.isnan(pfc)
        fpc[nan_mask] = np.nan + 0j
        
        surrogate_fpc.append(fpc)
    
    return surrogate_fpc


def compute_zhang_ppc_zscore(hpc_phases, pfc_phases, real_ppc, 
                              n_surrogates=200, min_cycles=10, seed=42):
    """
    Compute surrogate Z-scored PPC.
    
    1. Shuffle PFC cycles n_surrogates times
    2. Compute PPC for each surrogate
    3. Z = (real_PPC - mean_surr) / std_surr
    
    Args:
        hpc_phases: list of per-cycle HPC binned phases
        pfc_phases: list of per-cycle PFC binned phases
        real_ppc: the real PPC matrix (n_freqs, bin_count)
        n_surrogates: number of shuffle iterations
        min_cycles: minimum cycles for PPC
        seed: random seed
    
    Returns:
        z_ppc: Z-scored PPC (n_freqs, bin_count)
        surr_mean: mean of surrogate PPC (n_freqs, bin_count)
        surr_std: std of surrogate PPC (n_freqs, bin_count)
        surr_ppcs: all surrogate PPC matrices (n_surrogates, n_freqs, bin_count)
    """
    if real_ppc is None:
        return None, None, None, None
    
    rng = np.random.default_rng(seed)
    n_freqs, bin_count = real_ppc.shape
    surr_ppcs = np.full((n_surrogates, n_freqs, bin_count), np.nan)
    
    for s in range(n_surrogates):
        if (s + 1) % 50 == 0:
            print(f"  Surrogate {s+1}/{n_surrogates}")
        surr_fpc = compute_surrogate_fpc(hpc_phases, pfc_phases, rng)
        surr_ppc, _ = compute_zhang_ppc(surr_fpc, min_cycles=min_cycles)
        if surr_ppc is not None:
            surr_ppcs[s] = surr_ppc
    
    surr_mean = np.nanmean(surr_ppcs, axis=0)
    surr_std = np.nanstd(surr_ppcs, axis=0)
    
    # Avoid division by zero
    surr_std[surr_std == 0] = 1e-16
    
    z_ppc = (real_ppc - surr_mean) / surr_std
    
    return z_ppc, surr_mean, surr_std, surr_ppcs

print("Surrogate Z-scoring functions defined.")
print("  - compute_surrogate_fpc()")
print("  - compute_zhang_ppc_zscore()")

In [ ]:
# --- Compute surrogate Z-scored PPC for phasic and tonic (grand level) ---
n_surrogates = 200

print(f"Computing phasic surrogate Z-score ({n_surrogates} iterations, "
      f"{zhang_results['total_phasic_cycles']} cycles)...")
phasic_z_ppc, phasic_surr_mean, phasic_surr_std, phasic_surr_ppcs = compute_zhang_ppc_zscore(
    zhang_results['all_phasic_hpc_phases'],
    zhang_results['all_phasic_pfc_phases'],
    zhang_results['grand_phasic_ppc'],
    n_surrogates=n_surrogates, min_cycles=10)

print(f"\nComputing tonic surrogate Z-score ({n_surrogates} iterations, "
      f"{zhang_results['total_tonic_cycles']} cycles)...")
tonic_z_ppc, tonic_surr_mean, tonic_surr_std, tonic_surr_ppcs = compute_zhang_ppc_zscore(
    zhang_results['all_tonic_hpc_phases'],
    zhang_results['all_tonic_pfc_phases'],
    zhang_results['grand_tonic_ppc'],
    n_surrogates=n_surrogates, min_cycles=10)

if phasic_z_ppc is not None:
    print(f"\nPhasic Z-PPC range: [{np.nanmin(phasic_z_ppc):.2f}, {np.nanmax(phasic_z_ppc):.2f}]")
if tonic_z_ppc is not None:
    print(f"Tonic Z-PPC range: [{np.nanmin(tonic_z_ppc):.2f}, {np.nanmax(tonic_z_ppc):.2f}]")

In [ ]:
# --- Plot: Z-scored PPC heatmaps (phasic, tonic, difference) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Shared colorbar for phasic & tonic Z-PPC
vmin_z, vmax_z = None, None
for zppc in [phasic_z_ppc, tonic_z_ppc]:
    if zppc is not None:
        fmin, fmax = np.nanmin(zppc), np.nanmax(zppc)
        vmin_z = fmin if vmin_z is None else min(vmin_z, fmin)
        vmax_z = fmax if vmax_z is None else max(vmax_z, fmax)

for ax, zppc, label, n_cyc in zip(axes[:2],
                                   [phasic_z_ppc, tonic_z_ppc],
                                   ['Phasic', 'Tonic'],
                                   [zhang_results['total_phasic_cycles'],
                                    zhang_results['total_tonic_cycles']]):
    if zppc is None:
        ax.set_title(f'{label}: no data')
        continue
    
    im = ax.imshow(zppc, aspect='auto', origin='lower', interpolation='bilinear',
                   extent=[-180, 180, frequencies[0], frequencies[-1]],
                   cmap='hot', vmin=vmin_z, vmax=vmax_z)
    
    # Mode frequency from Z-scored PPC
    zppc_per_freq = np.nanmax(zppc, axis=1)
    mode_idx = int(np.nanargmax(zppc_per_freq))
    mode_freq = float(frequencies[mode_idx])
    ax.axhline(mode_freq, color='cyan', linestyle='--', linewidth=1.5,
               label=f'Mode {mode_freq:.1f} Hz')
    
    plt.colorbar(im, ax=ax, label='Z-score')
    ax.set_xlabel('Theta Phase (deg)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(f'{label} REM | Z-scored PPC (N={n_cyc}, {n_surrogates} surr)')
    ax.legend(loc='upper right', frameon=False)

# Difference
if phasic_z_ppc is not None and tonic_z_ppc is not None:
    ax = axes[2]
    diff = phasic_z_ppc - tonic_z_ppc
    vmax_diff = np.nanmax(np.abs(diff))
    im = ax.imshow(diff, aspect='auto', origin='lower', interpolation='bilinear',
                   extent=[-180, 180, frequencies[0], frequencies[-1]],
                   cmap='RdBu_r', vmin=-vmax_diff, vmax=vmax_diff)
    plt.colorbar(im, ax=ax, label='Z-score difference')
    ax.set_xlabel('Theta Phase (deg)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title('Phasic - Tonic (Z-scored)')

plt.suptitle('Surrogate Z-Scored PPC: HPC-PFC Gamma', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot: Z-scored PPC spectrum (averaged over theta phases) ---
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

if phasic_z_ppc is not None:
    phasic_z_spec = np.nanmean(phasic_z_ppc, axis=1)
    ax.plot(frequencies, phasic_z_spec, color='crimson', linewidth=2,
            label=f'Phasic (N={zhang_results["total_phasic_cycles"]})')

if tonic_z_ppc is not None:
    tonic_z_spec = np.nanmean(tonic_z_ppc, axis=1)
    ax.plot(frequencies, tonic_z_spec, color='steelblue', linewidth=2,
            label=f'Tonic (N={zhang_results["total_tonic_cycles"]})')

ax.axhline(0, color='gray', linestyle=':', alpha=0.5, label='Chance (Z=0)')
ax.axhline(1.96, color='gray', linestyle='--', alpha=0.3, label='Z=1.96 (p<0.05)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Z-scored PPC')
ax.set_title(f'Surrogate Z-Scored PPC Spectrum: HPC-PFC ({n_surrogates} surrogates)')
ax.legend()
plt.tight_layout()
plt.show()

## Section E: Statistics — Phasic vs Tonic PPC Comparison

Following Zhang et al.: compare PPC across conditions using per-rat PPC spectra as independent observations. Test at each frequency with paired t-tests (or Wilcoxon if few rats), FDR-corrected.

In [ ]:
from scipy.stats import ttest_rel, wilcoxon
from statsmodels.stats.multitest import multipletests

def compare_ppc_phasic_tonic(per_rat_phasic_ppc, per_rat_tonic_ppc, frequencies, 
                              alpha=0.05, method='fdr_bh'):
    """
    Statistical comparison of phasic vs tonic Zhang PPC across rats.
    
    For each frequency, compute the mean PPC (averaged over theta phases) per rat,
    then run a paired test across rats.
    
    Args:
        per_rat_phasic_ppc: dict rat_id -> (n_freqs, bin_count) PPC matrix
        per_rat_tonic_ppc: dict rat_id -> (n_freqs, bin_count) PPC matrix
        frequencies: frequency vector
        alpha: significance level
        method: multiple comparison correction ('fdr_bh', 'bonferroni', etc.)
    
    Returns:
        stats_df: DataFrame with frequency, phasic_mean, tonic_mean, p_value, q_value, significant
    """
    # Find rats that have both phasic and tonic
    common_rats = sorted(set(per_rat_phasic_ppc.keys()) & set(per_rat_tonic_ppc.keys()))
    n_rats = len(common_rats)
    print(f"Rats with both phasic and tonic PPC: {common_rats} (n={n_rats})")
    
    if n_rats < 3:
        print("Need at least 3 rats for statistical comparison.")
        return None
    
    n_freqs = len(frequencies)
    
    # Extract per-rat PPC spectra (averaged over theta phases)
    phasic_spectra = np.zeros((n_rats, n_freqs))
    tonic_spectra = np.zeros((n_rats, n_freqs))
    
    for i, rid in enumerate(common_rats):
        phasic_spectra[i] = np.nanmean(per_rat_phasic_ppc[rid], axis=1)
        tonic_spectra[i] = np.nanmean(per_rat_tonic_ppc[rid], axis=1)
    
    # Paired test at each frequency
    p_values = np.ones(n_freqs)
    t_stats = np.zeros(n_freqs)
    
    for f in range(n_freqs):
        p_vals = phasic_spectra[:, f]
        t_vals = tonic_spectra[:, f]
        
        # Skip if all NaN
        valid = np.isfinite(p_vals) & np.isfinite(t_vals)
        if np.sum(valid) < 3:
            continue
        
        if n_rats >= 6:
            t_stats[f], p_values[f] = ttest_rel(p_vals[valid], t_vals[valid])
        else:
            # Wilcoxon for small samples
            try:
                t_stats[f], p_values[f] = wilcoxon(p_vals[valid], t_vals[valid])
            except ValueError:
                pass
    
    # FDR correction
    reject, q_values, _, _ = multipletests(p_values, alpha=alpha, method=method)
    
    stats_df = pd.DataFrame({
        'frequency': frequencies,
        'phasic_mean': np.nanmean(phasic_spectra, axis=0),
        'tonic_mean': np.nanmean(tonic_spectra, axis=0),
        'phasic_sem': np.nanstd(phasic_spectra, axis=0) / np.sqrt(n_rats),
        'tonic_sem': np.nanstd(tonic_spectra, axis=0) / np.sqrt(n_rats),
        'test_stat': t_stats,
        'p_value': p_values,
        'q_value': q_values,
        'significant': reject,
    })
    
    n_sig = reject.sum()
    print(f"Significant frequencies (q < {alpha}): {n_sig} / {n_freqs}")
    if n_sig > 0:
        sig_freqs = frequencies[reject]
        print(f"  Frequency range: {sig_freqs.min():.0f} - {sig_freqs.max():.0f} Hz")
    
    return stats_df, phasic_spectra, tonic_spectra, common_rats

print("compare_ppc_phasic_tonic defined.")

In [ ]:
# --- Run statistical comparison ---
stats_result = compare_ppc_phasic_tonic(
    zhang_results['per_rat_phasic_ppc'],
    zhang_results['per_rat_tonic_ppc'],
    frequencies, alpha=0.05, method='fdr_bh')

if stats_result is not None:
    stats_df, phasic_spectra, tonic_spectra, common_rats = stats_result
    print(f"\nPer-rat spectra shape: phasic={phasic_spectra.shape}, tonic={tonic_spectra.shape}")
    display(stats_df[stats_df['significant']].head(20))

In [ ]:
# --- Plot 5: PPC spectra with significance markers ---
if stats_result is not None:
    stats_df, phasic_spectra, tonic_spectra, common_rats = stats_result
    n_rats = len(common_rats)
    
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={'height_ratios': [3, 1]})
    
    # Top: PPC spectra with SEM and significance
    ax = axes[0]
    p_mean = stats_df['phasic_mean'].values
    t_mean = stats_df['tonic_mean'].values
    p_sem = stats_df['phasic_sem'].values
    t_sem = stats_df['tonic_sem'].values
    
    ax.plot(frequencies, p_mean, color='crimson', linewidth=2, label=f'Phasic (n={n_rats} rats)')
    ax.fill_between(frequencies, p_mean - p_sem, p_mean + p_sem, color='crimson', alpha=0.2)
    ax.plot(frequencies, t_mean, color='steelblue', linewidth=2, label=f'Tonic (n={n_rats} rats)')
    ax.fill_between(frequencies, t_mean - t_sem, t_mean + t_sem, color='steelblue', alpha=0.2)
    
    # Mark significant frequencies
    sig_mask = stats_df['significant'].values
    if np.any(sig_mask):
        ymax = max(np.nanmax(p_mean + p_sem), np.nanmax(t_mean + t_sem))
        ax.scatter(frequencies[sig_mask], np.full(sig_mask.sum(), ymax * 1.05), 
                   marker='*', color='black', s=30, zorder=5, label='q < 0.05 (FDR)')
    
    ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PPC (Zhang, inter-cycle)')
    ax.set_title('Zhang PPC: Phasic vs Tonic (mean ± SEM across rats)')
    ax.legend()
    
    # Bottom: q-values
    ax = axes[1]
    ax.semilogy(frequencies, stats_df['q_value'].values, color='black', linewidth=1)
    ax.axhline(0.05, color='red', linestyle='--', linewidth=1, label='q = 0.05')
    ax.axhline(0.01, color='orange', linestyle='--', linewidth=1, label='q = 0.01')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('q-value (FDR)')
    ax.set_title('FDR-corrected p-values')
    ax.legend(fontsize=8)
    ax.set_ylim([1e-4, 1.1])
    
    plt.tight_layout()
    plt.show()